# Bài 7: Tìm kiếm biên tối ưu Pareto cho bài toán phân bổ ngân sách đa mục tiêu
Chào mừng bạn đến với hệ thống mô hình hỗ trợ ra quyết định kinh tế vĩ mô Việt Nam trong kỷ nguyên AI (**AIDEOM-VN**).
Tệp Notebook này hoàn toàn tự chứa (self-contained) và có thể chạy trực tiếp trên **Google Colab** hoặc **Kaggle**.

In [ ]:
# ==============================================================================
# HƯỚNG DẪN CHẠY TRÊN GOOGLE COLAB / KAGGLE (TỰ CHỨA - SELF-CONTAINED)
# Bằng cách chạy ô này, toàn bộ môi trường AIDEOM-VN sẽ được cấu hình tự động.
# ==============================================================================

import os
import sys

print("Đang cấu hình môi trường chạy trên Google Colab / Kaggle...")

# 1. Cài đặt các thư viện cần thiết
!pip install -q pulp cvxpy gymnasium numpy pandas matplotlib scipy seaborn plotly stable-baselines3


# Try to install optional libraries
try:
    !pip install -q kaleido
except:
    pass  # Optional, plotly still works

# 2. Tạo cấu trúc thư mục dự án
os.makedirs("aideom_vn/data", exist_ok=True)
os.makedirs("aideom_vn/src", exist_ok=True)

# 3. Tạo các tệp dữ liệu CSV trên hệ thống tệp Colab/Kaggle
macro_csv_data = 'year,GDP_trillion_VND,K_trillion_VND,L_million,D_digital_pct,AI_tech_firms_thousand,H_trained_pct\n2020,8044.4,16500,53.6,12.0,55.6,24.1\n2021,8487.5,17800,50.5,12.7,60.2,26.1\n2022,9513.3,19600,51.7,14.3,65.4,26.2\n2023,10221.8,21300,52.4,16.5,67.0,27.0\n2024,11511.9,23500,52.9,18.3,73.8,28.4\n2025,12847.6,25900,53.4,19.5,80.1,29.2\n'
priorities_csv_data = 'weight_id,a1_growth,a2_productivity,a3_spillover,a4_export,a5_employment,a6_ai_readiness,a7_risk,label\ndefault,0.15,0.15,0.20,0.15,0.10,0.20,0.15,Default\ngrowth,0.25,0.20,0.15,0.20,0.05,0.10,0.05,Growth-Oriented\ninclusive,0.10,0.10,0.25,0.05,0.25,0.10,0.15,Inclusive\n'
regions_csv_data = 'region_id,region_name_vi,grdp_per_capita_million_VND,fdi_registered_billion_USD,digital_index_0_100,ai_readiness_0_100,trained_labor_pct,rd_intensity_pct,internet_penetration_pct,gini_coef\n1,Trung du miền núi phía Bắc,57.0,3.5,38,22,21.5,0.18,72,0.405\n2,Đồng bằng sông Hồng,152.3,20.0,78,68,36.8,0.85,92,0.358\n3,Bắc Trung Bộ và Duyên hải miền Trung,87.5,8.2,55,40,27.5,0.32,84,0.372\n4,Tây Nguyên,68.9,0.8,32,18,18.2,0.15,68,0.412\n5,Đông Nam Bộ,158.9,18.5,82,75,42.5,0.78,94,0.385\n6,Đồng bằng sông Cửu Long,80.5,2.1,48,30,16.8,0.22,78,0.392\n'
sectors_csv_data = 'sector_id,sector_name_vi,growth_rate_2024_pct,productivity_million_VND_per_worker,spillover_coef_0_1,export_billion_USD,labor_million,ai_readiness_0_100,automation_risk_pct\n1,Nông-Lâm-Thủy sản,3.27,103.4,0.35,40.5,13.20,15,18\n2,CN chế biến chế tạo,9.64,241.2,0.78,290.9,11.50,55,42\n3,Xây dựng,7.45,168.8,0.42,2.5,4.80,20,25\n4,Khai khoáng,-1.20,1290.5,0.30,8.2,0.30,30,55\n5,Bán buôn-bán lẻ,7.10,145.3,0.55,5.5,7.80,48,38\n6,Tài chính-Ngân hàng,7.36,1072.4,0.85,1.2,0.55,72,52\n7,Logistics-Vận tải,9.93,321.4,0.72,3.1,1.95,42,35\n8,CNTT-Truyền thông,7.85,713.8,0.92,178.0,0.62,88,28\n9,Giáo dục-Đào tạo,6.42,205.7,0.65,0.0,2.15,38,22\n10,Y tế,6.85,437.1,0.60,0.0,0.75,45,18\n'

with open("aideom_vn/data/vietnam_macro_2020_2025.csv", "w", encoding="utf-8") as f:
    f.write(macro_csv_data)

with open("aideom_vn/data/vietnam_priorities.csv", "w", encoding="utf-8") as f:
    f.write(priorities_csv_data)

with open("aideom_vn/data/vietnam_regions_2024.csv", "w", encoding="utf-8") as f:
    f.write(regions_csv_data)

with open("aideom_vn/data/vietnam_sectors_2024.csv", "w", encoding="utf-8") as f:
    f.write(sectors_csv_data)

# 4. Tạo các tệp module python
data_loader_py_data = 'import os\nimport pandas as pd\nfrom pathlib import Path\nfrom typing import Dict, Any\n\n# Xác định đường dẫn tương đối đến thư mục data\nDATA_DIR = Path(__file__).resolve().parent.parent / \'data\'\n\ndef load_macro() -> pd.DataFrame:\n    """\n    Nạp dữ liệu vĩ mô Việt Nam giai đoạn 2020-2025.\n    \n    Returns:\n        pd.DataFrame: DataFrame chứa số liệu vĩ mô được sắp xếp theo năm.\n    """\n    file_path = DATA_DIR / \'vietnam_macro_2020_2025.csv\'\n    if not file_path.exists():\n        raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu vĩ mô tại: {file_path}")\n    df = pd.read_csv(file_path)\n    # Đảm bảo dữ liệu được sắp xếp tăng dần theo năm\n    df = df.sort_values(\'year\').reset_index(drop=True)\n    return df\n\ndef load_sectors() -> pd.DataFrame:\n    """\n    Nạp dữ liệu 10 ngành kinh tế chính của Việt Nam năm 2024.\n    \n    Returns:\n        pd.DataFrame: DataFrame chứa số liệu cấu trúc ngành.\n    """\n    file_path = DATA_DIR / \'vietnam_sectors_2024.csv\'\n    if not file_path.exists():\n        raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu ngành tại: {file_path}")\n    return pd.read_csv(file_path)\n\ndef load_regions() -> pd.DataFrame:\n    """\n    Nạp dữ liệu 6 vùng kinh tế - xã hội của Việt Nam năm 2024.\n    \n    Returns:\n        pd.DataFrame: DataFrame chứa số liệu vùng miền.\n    """\n    file_path = DATA_DIR / \'vietnam_regions_2024.csv\'\n    if not file_path.exists():\n        raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu vùng tại: {file_path}")\n    return pd.read_csv(file_path)\n\ndef load_priorities() -> pd.DataFrame:\n    """\n    Nạp dữ liệu trọng số ưu tiên chính sách phát triển.\n    \n    Returns:\n        pd.DataFrame: DataFrame chứa 3 bộ trọng số ưu tiên.\n    """\n    file_path = DATA_DIR / \'vietnam_priorities.csv\'\n    if not file_path.exists():\n        raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu trọng số tại: {file_path}")\n    return pd.read_csv(file_path)\n\ndef validate_data() -> None:\n    """\n    Thực hiện kiểm tra và in ra hình dạng cũng như dòng dữ liệu đầu tiên của mỗi tệp.\n    """\n    print("--- BẮT ĐẦU XÁC THỰC DỮ LIỆU ---")\n    \n    # Kiểm tra tệp vĩ mô\n    macro = load_macro()\n    print(f"\\nTệp vĩ mô (vietnam_macro_2020_2025.csv) - Kích thước: {macro.shape}")\n    print(macro.head(2))\n    \n    # Kiểm tra tệp ngành\n    sectors = load_sectors()\n    print(f"\\nTệp ngành (vietnam_sectors_2024.csv) - Kích thước: {sectors.shape}")\n    print(sectors.head(2))\n    \n    # Kiểm tra tệp vùng miền\n    regions = load_regions()\n    print(f"\\nTệp vùng (vietnam_regions_2024.csv) - Kích thước: {regions.shape}")\n    print(regions.head(2))\n    \n    # Kiểm tra tệp trọng số\n    priorities = load_priorities()\n    print(f"\\nTệp trọng số (vietnam_priorities.csv) - Kích thước: {priorities.shape}")\n    print(priorities.head(2))\n    \n    print("\\n--- XÁC THỰC DỮ LIỆU THÀNH CÔNG ---")\n\nif __name__ == \'__main__\':\n    # Chạy kiểm tra dữ liệu trực tiếp khi gọi module\n    validate_data()\n'
m1_production_py_data = 'import numpy as np\nimport pandas as pd\nfrom typing import Dict, Any, Union\n\ndef compute_tfp(\n    Y: Union[float, np.ndarray],\n    K: Union[float, np.ndarray],\n    L: Union[float, np.ndarray],\n    D: Union[float, np.ndarray],\n    AI: Union[float, np.ndarray],\n    H: Union[float, np.ndarray],\n    alpha: float = 0.33,\n    beta: float = 0.42,\n    gamma: float = 0.10,\n    delta: float = 0.08,\n    theta: float = 0.07\n) -> Union[float, np.ndarray]:\n    """\n    Tính toán Năng suất Nhân tố Tổng hợp (TFP - A_t) từ hàm sản xuất Cobb-Douglas mở rộng.\n    A_t = Y / (K^α · L^β · D^γ · AI^δ · H^θ)\n    \n    Args:\n        Y: GDP (nghìn tỷ VND)\n        K: Vốn lũy kế (nghìn tỷ VND)\n        L: Lao động (triệu người)\n        D: Chỉ số chuyển đổi số D (%)\n        AI: Năng lực AI (nghìn DN công nghệ số)\n        H: Vốn nhân lực đào tạo (%)\n        alpha: Độ co giãn của vốn vật chất.\n        beta: Độ co giãn của lao động.\n        gamma: Độ co giãn của số hóa.\n        delta: Độ co giãn của AI.\n        theta: Độ co giãn của nhân lực.\n        \n    Returns:\n        Union[float, np.ndarray]: TFP A_t tương ứng cho từng điểm dữ liệu.\n    """\n    # Giải ngược hàm Cobb-Douglas để tìm TFP A_t\n    denominator = (K ** alpha) * (L ** beta) * (D ** gamma) * (AI ** delta) * (H ** theta)\n    return Y / denominator\n\ndef forecast_gdp(\n    A_mean: float,\n    K: Union[float, np.ndarray],\n    L: Union[float, np.ndarray],\n    D: Union[float, np.ndarray],\n    AI: Union[float, np.ndarray],\n    H: Union[float, np.ndarray],\n    alpha: float = 0.33,\n    beta: float = 0.42,\n    gamma: float = 0.10,\n    delta: float = 0.08,\n    theta: float = 0.07\n) -> Union[float, np.ndarray]:\n    """\n    Dự báo sản lượng GDP (Ŷ) bằng hàm Cobb-Douglas mở rộng dựa trên giá trị TFP trung bình.\n    Ŷ = A_mean · K^α · L^β · D^γ · AI^δ · H^θ\n    \n    Args:\n        A_mean: TFP trung bình hoặc xu hướng dự tính.\n        K, L, D, AI, H: Các yếu tố đầu vào tương ứng.\n        \n    Returns:\n        Union[float, np.ndarray]: GDP dự báo (nghìn tỷ VND).\n    """\n    # Tính GDP dự báo Ŷ\n    return A_mean * (K ** alpha) * (L ** beta) * (D ** gamma) * (AI ** delta) * (H ** theta)\n\ndef mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:\n    """\n    Tính toán sai số phần trăm tuyệt đối trung bình (MAPE) giữa giá trị thực tế và dự báo.\n    \n    Args:\n        y_true: Giá trị thực tế.\n        y_pred: Giá trị dự báo.\n        \n    Returns:\n        float: Sai số phần trăm (%) của mô hình dự báo.\n    """\n    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)\n\ndef growth_decomposition(df: pd.DataFrame, params: Dict[str, float]) -> pd.DataFrame:\n    """\n    Phân rã đóng góp tăng trưởng kinh tế (Growth Accounting):\n    Δln(Y) = Δln(A) + α·Δln(K) + β·Δln(L) + γ·Δln(D) + δ·Δln(AI) + θ·Δln(H)\n    \n    Args:\n        df: DataFrame chứa các cột \'year\', \'GDP_trillion_VND\', \'K_trillion_VND\', \'L_million\',\n            \'D_digital_pct\', \'AI_tech_firms_thousand\', \'H_trained_pct\'.\n        params: Các tham số alpha, beta, gamma, delta, theta.\n        \n    Returns:\n        pd.DataFrame: Bảng kết quả phân rã đóng góp tăng trưởng của từng yếu tố qua các năm.\n    """\n    alpha = params.get(\'alpha\', 0.33)\n    beta = params.get(\'beta\', 0.42)\n    gamma = params.get(\'gamma\', 0.10)\n    delta = params.get(\'delta\', 0.08)\n    theta = params.get(\'theta\', 0.07)\n    \n    # Tính TFP cho mỗi năm trước tiên\n    A = compute_tfp(\n        df[\'GDP_trillion_VND\'].values, df[\'K_trillion_VND\'].values, df[\'L_million\'].values,\n        df[\'D_digital_pct\'].values, df[\'AI_tech_firms_thousand\'].values, df[\'H_trained_pct\'].values,\n        alpha, beta, gamma, delta, theta\n    )\n    df_calc = df.copy()\n    df_calc[\'A\'] = A\n    \n    # Tính logarit tự nhiên cho tất cả các biến số\n    log_cols = {\n        \'ln_Y\': np.log(df_calc[\'GDP_trillion_VND\']),\n        \'ln_A\': np.log(df_calc[\'A\']),\n        \'ln_K\': np.log(df_calc[\'K_trillion_VND\']),\n        \'ln_L\': np.log(df_calc[\'L_million\']),\n        \'ln_D\': np.log(df_calc[\'D_digital_pct\']),\n        \'ln_AI\': np.log(df_calc[\'AI_tech_firms_thousand\']),\n        \'ln_H\': np.log(df_calc[\'H_trained_pct\'])\n    }\n    \n    # Tính sai phân (tốc độ tăng trưởng liên hoàn)\n    diffs = {}\n    for col_name, log_val in log_cols.items():\n        diffs[col_name] = log_val.diff()\n        \n    # Tạo bảng kết quả phân rã\n    decomp = pd.DataFrame()\n    decomp[\'year\'] = df_calc[\'year\'][1:] # Bỏ năm đầu tiên do sai phân\n    decomp[\'g_GDP\'] = diffs[\'ln_Y\'][1:]\n    \n    # Đóng góp của từng yếu tố đầu vào\n    decomp[\'contrib_TFP\'] = diffs[\'ln_A\'][1:]\n    decomp[\'contrib_K\'] = alpha * diffs[\'ln_K\'][1:]\n    decomp[\'contrib_L\'] = beta * diffs[\'ln_L\'][1:]\n    decomp[\'contrib_D\'] = gamma * diffs[\'ln_D\'][1:]\n    decomp[\'contrib_AI\'] = delta * diffs[\'ln_AI\'][1:]\n    decomp[\'contrib_H\'] = theta * diffs[\'ln_H\'][1:]\n    \n    # Đưa về dạng tỷ trọng đóng góp (%) so với tổng tăng trưởng GDP\n    factor_cols = [\'contrib_TFP\', \'contrib_K\', \'contrib_L\', \'contrib_D\', \'contrib_AI\', \'contrib_H\']\n    for col in factor_cols:\n        decomp[col + \'_share_pct\'] = (decomp[col] / decomp[\'g_GDP\']) * 100\n        \n    return decomp.reset_index(drop=True)\n\ndef scenario_2030(\n    A_trend: float,\n    K0: float = 25900,\n    L0: float = 53.4,\n    D_target: float = 30.0,\n    AI_target: float = 100.0,\n    H_target: float = 35.0,\n    K_growth: float = 0.06,\n    L_growth: float = 0.06,\n    TFP_growth: float = 0.012,\n    alpha: float = 0.33,\n    beta: float = 0.42,\n    gamma: float = 0.10,\n    delta: float = 0.08,\n    theta: float = 0.07\n) -> Dict[str, float]:\n    """\n    Mô phỏng và dự báo GDP của Việt Nam năm 2030 dưới các giả định kịch bản.\n    \n    Args:\n        A_trend: TFP cơ sở tại năm 2025.\n        K0, L0: Vốn và lao động cơ sở năm 2025.\n        D_target, AI_target, H_target: Mục tiêu kịch bản số hóa, AI và nhân lực năm 2030.\n        K_growth, L_growth: Tốc độ tăng trưởng hàng năm của vốn và lao động từ 2025-2030.\n        TFP_growth: Tốc độ tăng trưởng hàng năm của TFP từ 2025-2030.\n        \n    Returns:\n        Dict[str, float]: Kết quả mô phỏng các giá trị K, L, D, AI, H, A và GDP (Y) năm 2030.\n    """\n    n_years = 5 # Từ năm 2025 đến năm 2030 là 5 năm\n    \n    # Tích lũy vốn và phát triển lao động qua các năm\n    K_2030 = K0 * ((1 + K_growth) ** n_years)\n    L_2030 = L0 * ((1 + L_growth) ** n_years)\n    \n    # Tăng trưởng TFP theo xu hướng công nghệ\n    A_2030 = A_trend * ((1 + TFP_growth) ** n_years)\n    \n    # Tính GDP dự báo Y_2030\n    Y_2030 = forecast_gdp(\n        A_mean=A_2030, K=K_2030, L=L_2030, D=D_target, AI=AI_target, H=H_target,\n        alpha=alpha, beta=beta, gamma=gamma, delta=delta, theta=theta\n    )\n    \n    return {\n        \'K_2030\': K_2030,\n        \'L_2030\': L_2030,\n        \'D_2030\': D_target,\n        \'AI_2030\': AI_target,\n        \'H_2030\': H_target,\n        \'A_2030\': A_2030,\n        \'GDP_2030\': Y_2030\n    }\n'
m2_readiness_py_data = 'import numpy as np\nimport pandas as pd\nfrom typing import Tuple, List, Union\n\ndef topsis(\n    X: np.ndarray,\n    weights: np.ndarray,\n    is_benefit: Union[List[bool], np.ndarray]\n) -> Tuple[np.ndarray, np.ndarray]:\n    """\n    Hiện thực thuật toán TOPSIS từ đầu bằng NumPy.\n    \n    Args:\n        X: Ma trận quyết định ban đầu kích thước (n_alternatives, m_criteria).\n        weights: Vector trọng số của các tiêu chí (tổng bằng 1).\n        is_benefit: List hoặc vector Boolean xác định tiêu chí là lợi ích (True) hay chi phí (False).\n        \n    Returns:\n        Tuple[np.ndarray, np.ndarray]:\n            - C_star: Điểm gần gũi tương đối (closeness coefficient) trong đoạn [0, 1].\n            - ranks: Thứ hạng của các phương án (1 là tốt nhất, n là kém nhất).\n    """\n    n, m = X.shape\n    is_benefit = np.array(is_benefit)\n    weights = np.array(weights)\n    \n    # Bước 1: Chuẩn hóa vector ma trận quyết định\n    # r_ij = x_ij / sqrt(sum_i(x_ij^2))\n    norm_denom = np.sqrt(np.sum(X ** 2, axis=0))\n    # Tránh chia cho 0 nếu một cột toàn số 0\n    norm_denom = np.where(norm_denom == 0, 1e-12, norm_denom)\n    R = X / norm_denom\n    \n    # Bước 2: Nhân với trọng số\n    # v_ij = w_j * r_ij\n    V = R * weights\n    \n    # Bước 3: Xác định phương án lý tưởng dương (A*) và lý tưởng âm (A-)\n    # A* = max(v_ij) cho tiêu chí tốt, min(v_ij) cho tiêu chí xấu\n    A_star = np.zeros(m)\n    A_neg = np.zeros(m)\n    \n    for j in range(m):\n        if is_benefit[j]:\n            A_star[j] = np.max(V[:, j])\n            A_neg[j] = np.min(V[:, j])\n        else:\n            A_star[j] = np.min(V[:, j])\n            A_neg[j] = np.max(V[:, j])\n            \n    # Bước 4: Tính khoảng cách Euclide từ mỗi phương án đến A* và A-\n    S_star = np.sqrt(np.sum((V - A_star) ** 2, axis=1))\n    S_neg = np.sqrt(np.sum((V - A_neg) ** 2, axis=1))\n    \n    # Bước 5: Tính hệ số tương đồng gần gũi C_star\n    C_star = S_neg / (S_star + S_neg + 1e-12) # Thêm epsilon để tránh chia cho 0\n    \n    # Xếp hạng giảm dần theo điểm số C_star\n    # np.argsort(-C_star) cho chỉ số sắp xếp giảm dần, dùng argsort lần 2 để lấy thứ hạng cụ thể\n    ranks = np.argsort(np.argsort(-C_star)) + 1\n    \n    return C_star, ranks\n\ndef entropy_weights(X: np.ndarray) -> np.ndarray:\n    """\n    Tính trọng số khách quan bằng phương pháp Entropy.\n    \n    Args:\n        X: Ma trận quyết định ban đầu kích thước (n_alternatives, m_criteria).\n        \n    Returns:\n        np.ndarray: Vector trọng số Entropy (tổng bằng 1.0).\n    """\n    n, m = X.shape\n    \n    # Chuẩn hóa ma trận quyết định theo tỉ lệ tổng cột\n    # p_ij = x_ij / sum_i(x_ij)\n    col_sums = X.sum(axis=0)\n    col_sums = np.where(col_sums == 0, 1e-12, col_sums)\n    P = X / col_sums\n    \n    # Tính Entropy cho mỗi tiêu chí j\n    # E_j = -k * sum_i(p_ij * ln(p_ij))\n    k = 1.0 / np.log(n) if n > 1 else 1.0\n    \n    # Tránh ln(0) bằng np.log(P + 1e-12)\n    E = -k * np.sum(P * np.log(P + 1e-12), axis=0)\n    \n    # Tính độ phân kỳ (diversity)\n    d = 1.0 - E\n    \n    # Tính trọng số chuẩn hóa\n    w = d / (np.sum(d) + 1e-12)\n    \n    return w\n\ndef sensitivity_ai_weight(\n    X: np.ndarray,\n    is_benefit: Union[List[bool], np.ndarray],\n    w_base: np.ndarray,\n    w_ai_idx: int,\n    w_ai_range: np.ndarray\n) -> pd.DataFrame:\n    """\n    Phân tích độ nhạy của trọng số: thay đổi trọng số AI (tiêu chí tại w_ai_idx)\n    trong phạm vi w_ai_range, renormalize các trọng số còn lại sao cho tổng bằng 1,\n    và tính toán sự thay đổi điểm số cũng như thứ hạng.\n    \n    Args:\n        X: Ma trận quyết định ban đầu.\n        is_benefit: List thuộc tính tốt/xấu.\n        w_base: Bộ trọng số chuyên gia gốc.\n        w_ai_idx: Vị trí của trọng số AI.\n        w_ai_range: Các giá trị trọng số AI để kiểm tra (ví dụ từ 0.05 đến 0.40).\n        \n    Returns:\n        pd.DataFrame: Bảng kết quả thay đổi thứ hạng tương ứng.\n    """\n    n_alternatives = X.shape[0]\n    m_criteria = X.shape[1]\n    \n    results = []\n    \n    for w_ai in w_ai_range:\n        # Tạo bộ trọng số mới\n        w_new = w_base.copy()\n        w_new[w_ai_idx] = w_ai\n        \n        # Chuẩn hóa lại các trọng số khác để tổng vẫn bằng 1.0\n        other_indices = [i for i in range(m_criteria) if i != w_ai_idx]\n        other_sum = np.sum(w_base[other_indices])\n        \n        if other_sum > 0:\n            scale = (1.0 - w_ai) / other_sum\n            w_new[other_indices] = w_base[other_indices] * scale\n        else:\n            w_new[other_indices] = (1.0 - w_ai) / len(other_indices)\n            \n        # Đảm bảo tổng trọng số bằng 1.0 tuyệt đối\n        w_new = w_new / np.sum(w_new)\n        \n        # Chạy TOPSIS với trọng số mới\n        scores, ranks = topsis(X, w_new, is_benefit)\n        \n        # Lưu kết quả\n        for alt_idx in range(n_alternatives):\n            results.append({\n                \'w_AI\': w_ai,\n                \'alternative_id\': alt_idx,\n                \'score\': scores[alt_idx],\n                \'rank\': ranks[alt_idx]\n            })\n            \n    return pd.DataFrame(results)\n'
m3_allocation_py_data = 'import numpy as np\nimport scipy.optimize as opt\nimport pulp\nimport cvxpy as cp\nfrom typing import Tuple, Dict, Any, List, Optional\n\ndef solve_lp_scipy(budget_total: float = 100.0) -> Tuple[np.ndarray, float]:\n    """\n    Giải bài toán phân bổ ngân sách đơn giản (Bài 2) bằng scipy.optimize.linprog.\n    \n    Args:\n        budget_total: Tổng ngân sách đầu tư (nghìn tỷ VND).\n        \n    Returns:\n        Tuple[np.ndarray, float]:\n            - x_opt: Vector đầu tư tối ưu cho 4 hạng mục (I, AI, H, R&D).\n            - Z_opt: GDP tăng thêm cực đại (nghìn tỷ VND).\n    """\n    # Hàm mục tiêu: max Z = 0.85x1 + 1.20x2 + 0.95x3 + 1.35x4\n    # scipy tối thiểu hóa, nên ta đảo dấu hệ số\n    c = [-0.85, -1.20, -0.95, -1.35]\n    \n    # Ràng buộc bất đẳng thức A_ub * x <= b_ub\n    # x1 + x2 + x3 + x4 <= budget_total\n    # -x1 <= -25 (x1 >= 25)\n    # -x2 <= -15 (x2 >= 15)\n    # -x3 <= -20 (x3 >= 20)\n    # -x4 <= -10 (x4 >= 10)\n    # x2 + x4 >= 0.35(x1+x2+x3+x4) -> 0.35x1 - 0.65x2 + 0.35x3 - 0.65x4 <= 0\n    A_ub = [\n        [1.0, 1.0, 1.0, 1.0],\n        [-1.0, 0.0, 0.0, 0.0],\n        [0.0, -1.0, 0.0, 0.0],\n        [0.0, 0.0, -1.0, 0.0],\n        [0.0, 0.0, 0.0, -1.0],\n        [0.35, -0.65, 0.35, -0.65]\n    ]\n    b_ub = [budget_total, -25.0, -15.0, -20.0, -10.0, 0.0]\n    \n    bounds = [(0, None)] * 4\n    \n    # Giải bằng phương pháp HiGHS hiệu suất cao\n    res = opt.linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method=\'highs\')\n    \n    if not res.success:\n        raise ValueError(f"Không tìm thấy lời giải khả thi cho LP SciPy: {res.message}")\n        \n    return res.x, -res.fun\n\ndef solve_lp_pulp(budget_total: float = 100.0) -> Tuple[np.ndarray, float, Dict[str, float]]:\n    """\n    Giải bài toán phân bổ ngân sách đơn giản (Bài 2) bằng PuLP và in ra giá đối ngẫu.\n    \n    Args:\n        budget_total: Tổng ngân sách đầu tư.\n        \n    Returns:\n        Tuple[np.ndarray, float, Dict[str, float]]:\n            - x_opt: Vector đầu tư tối ưu.\n            - Z_opt: GDP tăng cực đại.\n            - shadow_prices: Từ điển chứa giá đối ngẫu (shadow prices) của từng ràng buộc.\n    """\n    # Khởi tạo bài toán tối đa hóa\n    prob = pulp.LpProblem("Simple_Budget_Allocation", pulp.LpMaximize)\n    \n    # Biến quyết định\n    x1 = pulp.LpVariable(\'x1_Infra\', lowBound=0)\n    x2 = pulp.LpVariable(\'x2_AI\', lowBound=0)\n    x3 = pulp.LpVariable(\'x3_Human\', lowBound=0)\n    x4 = pulp.LpVariable(\'x4_RD\', lowBound=0)\n    \n    # Hàm mục tiêu\n    prob += 0.85*x1 + 1.20*x2 + 0.95*x3 + 1.35*x4, "Total_GDP_Gain"\n    \n    # Ràng buộc\n    c_budget = x1 + x2 + x3 + x4 <= budget_total\n    prob += c_budget, "Budget_Constraint"\n    \n    c_infra = x1 >= 25\n    prob += c_infra, "Min_Infra"\n    \n    c_ai = x2 >= 15\n    prob += c_ai, "Min_AI"\n    \n    c_human = x3 >= 20\n    prob += c_human, "Min_Human"\n    \n    c_rd = x4 >= 10\n    prob += c_rd, "Min_RD"\n    \n    c_strategic = x2 + x4 >= 0.35 * (x1 + x2 + x3 + x4)\n    prob += c_strategic, "Strategic_Tech_Ratio"\n    \n    # Giải bài toán bằng solver CBC mặc định của PuLP (đã đi kèm)\n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        raise ValueError(f"Không giải được LP tối ưu bằng PuLP: Status = {pulp.LpStatus[prob.status]}")\n        \n    x_opt = np.array([x1.varValue, x2.varValue, x3.varValue, x4.varValue])\n    Z_opt = pulp.value(prob.objective)\n    \n    # Trích xuất giá đối ngẫu (shadow prices) của các ràng buộc\n    shadow_prices = {\n        \'budget\': c_budget.pi,\n        \'min_infra\': c_infra.pi,\n        \'min_ai\': c_ai.pi,\n        \'min_human\': c_human.pi,\n        \'min_rd\': c_rd.pi,\n        \'strategic\': c_strategic.pi\n    }\n    \n    return x_opt, Z_opt, shadow_prices\n\ndef solve_allocation_pulp(\n    budget: float = 50000.0,\n    gamma: float = 0.002,\n    lam: float = 0.7,\n    with_fairness: bool = True\n) -> Tuple[np.ndarray, float]:\n    """\n    Giải bài toán phân bổ ngân sách 6 vùng x 4 hạng mục (Bài 4) bằng PuLP.\n    \n    Args:\n        budget: Tổng ngân sách đầu tư vùng miền.\n        gamma: Hệ số cải thiện số hóa trên mỗi tỷ đầu tư.\n        lam: Tham số mức độ công bằng (lambda in [0, 1]).\n        with_fairness: Nếu True, áp dụng ràng buộc công bằng vùng miền C5.\n        \n    Returns:\n        Tuple[np.ndarray, float]:\n            - x_matrix: Ma trận đầu tư tối ưu (6 vùng x 4 hạng mục).\n            - Z_opt: GDP tăng cực đại vùng miền (tỷ VND).\n    """\n    regions = [\'NMM\', \'RRD\', \'NCC\', \'CH\', \'SE\', \'MD\']\n    items = [\'I\', \'D\', \'AI\', \'H\']\n    \n    # Bảng hệ số tác động biên beta\n    beta = {\n        (\'NMM\',\'I\'):1.15,(\'NMM\',\'D\'):0.85,(\'NMM\',\'AI\'):0.55,(\'NMM\',\'H\'):1.30,\n        (\'RRD\',\'I\'):0.95,(\'RRD\',\'D\'):1.25,(\'RRD\',\'AI\'):1.40,(\'RRD\',\'H\'):1.05,\n        (\'NCC\',\'I\'):1.05,(\'NCC\',\'D\'):0.95,(\'NCC\',\'AI\'):0.85,(\'NCC\',\'H\'):1.15,\n        (\'CH\',\'I\') :1.20,(\'CH\',\'D\') :0.75,(\'CH\',\'AI\') :0.45,(\'CH\',\'H\') :1.35,\n        (\'SE\',\'I\') :0.90,(\'SE\',\'D\') :1.30,(\'SE\',\'AI\') :1.55,(\'SE\',\'H\') :1.00,\n        (\'MD\',\'I\') :1.10,(\'MD\',\'D\') :0.85,(\'MD\',\'AI\') :0.65,(\'MD\',\'H\') :1.25\n    }\n    \n    # Chỉ số số hóa ban đầu D_0\n    D0 = {\'NMM\':38.0, \'RRD\':78.0, \'NCC\':55.0, \'CH\':32.0, \'SE\':82.0, \'MD\':48.0}\n    \n    prob = pulp.LpProblem("VN_Region_Budget", pulp.LpMaximize)\n    \n    # Biến quyết định\n    x = pulp.LpVariable.dicts(\'x\', (regions, items), lowBound=0)\n    \n    # Hàm mục tiêu\n    prob += pulp.lpSum(beta[(r,j)] * x[r][j] for r in regions for j in items)\n    \n    # Ràng buộc\n    # C1: Ngân sách tổng\n    prob += pulp.lpSum(x[r][j] for r in regions for j in items) <= budget\n    \n    # C2 & C3: Ngưỡng sàn/trần của từng vùng miền\n    for r in regions:\n        prob += pulp.lpSum(x[r][j] for j in items) >= 5000\n        prob += pulp.lpSum(x[r][j] for j in items) <= 12000\n        \n    # C4: Sàn đào tạo nhân lực số tổng thể\n    prob += pulp.lpSum(x[r][\'H\'] for r in regions) >= 12000\n    \n    # C5: Ràng buộc công bằng vùng (Linearized bằng biến phụ M)\n    if with_fairness:\n        M = pulp.LpVariable(\'Dmax\')\n        for r in regions:\n            prob += D0[r] + gamma * x[r][\'D\'] <= M\n        for r in regions:\n            prob += D0[r] + gamma * x[r][\'D\'] >= lam * M\n            \n    # Giải LP\n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        raise ValueError("Không tìm thấy nghiệm tối ưu cho Bài 4 LP PuLP.")\n        \n    # Chuyển kết quả về dạng ma trận numpy 6x4\n    res_matrix = np.zeros((6, 4))\n    for r_idx, r in enumerate(regions):\n        for j_idx, j in enumerate(items):\n            res_matrix[r_idx, j_idx] = x[r][j].varValue\n            \n    return res_matrix, pulp.value(prob.objective)\n\ndef solve_allocation_cvxpy(\n    budget: float = 50000.0,\n    gamma: float = 0.002,\n    lam: float = 0.7,\n    with_fairness: bool = True\n) -> Tuple[np.ndarray, float]:\n    """\n    Giải bài toán phân bổ ngân sách 6 vùng x 4 hạng mục (Bài 4) bằng CVXPY.\n    \n    Returns:\n        Tuple[np.ndarray, float]:\n            - x_matrix: Ma trận đầu tư tối ưu (6 vùng x 4 hạng mục).\n            - Z_opt: GDP tăng cực đại vùng miền (tỷ VND).\n    """\n    # Ma trận beta: 6 vùng x 4 hạng mục\n    beta_mat = np.array([\n        [1.15, 0.85, 0.55, 1.30], # NMM\n        [0.95, 1.25, 1.40, 1.05], # RRD\n        [1.05, 0.95, 0.85, 1.15], # NCC\n        [1.20, 0.75, 0.45, 1.35], # CH\n        [0.90, 1.30, 1.55, 1.00], # SE\n        [1.10, 0.85, 0.65, 1.25]  # MD\n    ])\n    \n    D0_vec = np.array([38.0, 78.0, 55.0, 32.0, 82.0, 48.0])\n    \n    # Biến quyết định trong CVXPY\n    X = cp.Variable((6, 4), nonneg=True)\n    \n    # Hàm mục tiêu\n    objective = cp.Maximize(cp.sum(cp.multiply(beta_mat, X)))\n    \n    constraints = []\n    # C1: Ngân sách tổng\n    constraints.append(cp.sum(X) <= budget)\n    \n    # C2 & C3: Sàn/Trần đầu tư mỗi vùng\n    regional_sums = cp.sum(X, axis=1)\n    constraints.append(regional_sums >= 5000)\n    constraints.append(regional_sums <= 12000)\n    \n    # C4: Sàn đào tạo nhân lực số tổng thể\n    constraints.append(cp.sum(X[:, 3]) >= 12000)\n    \n    # C5: Ràng buộc công bằng vùng miền\n    if with_fairness:\n        # Trong CVXPY ta dùng cp.max để biểu diễn tối đa trực tiếp\n        # D_final_vec = D0_vec + gamma * X[:, 1]\n        D_final = D0_vec + gamma * X[:, 1]\n        max_D = cp.max(D_final)\n        \n        # Ràng buộc: D_final_r >= lam * max_D cho mọi r\n        for r in range(6):\n            constraints.append(D_final[r] >= lam * max_D)\n            \n    # Giải bài toán tối ưu lồi\n    prob = cp.Problem(objective, constraints)\n    prob.solve(solver=cp.SCS)\n    \n    if prob.status not in ["optimal", "optimal_inaccurate"]:\n        raise ValueError(f"Không giải được LP bằng CVXPY: Status = {prob.status}")\n        \n    return X.value, prob.value\n\ndef solve_project_selection(\n    budget_total: float = 80000.0,\n    budget_12: float = 40000.0,\n    force_p1_p2: bool = False,\n    use_risk: bool = False,\n    probabilities_dict: Optional[Dict[int, float]] = None\n) -> Tuple[List[int], float, float]:\n    """\n    Giải bài toán quy hoạch nguyên MIP lựa chọn dự án chuyển đổi số (Bài 5) bằng PuLP.\n    \n    Args:\n        budget_total: Tổng ngân sách 5 năm (tỷ VND).\n        budget_12: Ngân sách tối đa cho năm 1-2 (tỷ VND).\n        force_p1_p2: Nếu True, bỏ qua ràng buộc loại trừ C3 và ép chọn cả P1 và P2.\n        use_risk: Nếu True, tối ưu hóa lợi ích kỳ vọng E[Z] = sum(p_i * B_i * y_i).\n        probabilities_dict: Từ điển chứa xác suất thành công p_i của mỗi dự án.\n        \n    Returns:\n        Tuple[List[int], float, float]:\n            - selected_projects: Danh sách các dự án được chọn (1-indexed).\n            - total_cost: Tổng chi phí thực tế của các dự án được chọn.\n            - total_benefit: Tổng lợi ích (hoặc lợi ích kỳ vọng) Z*.\n    """\n    P = list(range(1, 16))\n    \n    # Bảng chi phí C, chi phí 2 năm đầu C1, và lợi ích B của 15 dự án\n    C = {1:12000, 2:11500, 3:18000, 4:4500, 5:3200, 6:5800, 7:6500, 8:15000,\n         9:2500, 10:7200, 11:4800, 12:8500, 13:20000, 14:3800, 15:1500}\n         \n    C1 = {1:8500, 2:7500, 3:12000, 4:3500, 5:2500, 6:4000, 7:4500, 8:9000,\n          9:1800, 10:5000, 11:3500, 12:5500, 13:13000, 14:2800, 15:1200}\n          \n    B = {1:21500, 2:20800, 3:32500, 4:9200, 5:6800, 6:11400, 7:12200, 8:28500,\n         9:5800, 10:13800, 11:8500, 12:16200, 13:35000, 14:7500, 15:3800}\n         \n    # Khởi tạo bài toán MIP\n    prob = pulp.LpProblem("VN_Project_Selection", pulp.LpMaximize)\n    \n    # Biến quyết định nhị phân\n    y = pulp.LpVariable.dicts(\'y\', P, cat=\'Binary\')\n    \n    # Xác định hàm mục tiêu\n    if use_risk:\n        if probabilities_dict is None:\n            # Xác suất mặc định nếu không truyền\n            probabilities_dict = {\n                1:0.85, 2:0.85, 3:0.85, # Hạ tầng\n                4:0.75, 5:0.75,         # Chính phủ\n                8:0.65, 12:0.65, 13:0.65, # AI & bán dẫn\n                6:0.80, 7:0.80, 9:0.80, 10:0.80, 11:0.80, 14:0.80, 15:0.80 # Còn lại\n            }\n        prob += pulp.lpSum(probabilities_dict[i] * B[i] * y[i] for i in P), "Expected_Total_Benefit"\n    else:\n        prob += pulp.lpSum(B[i] * y[i] for i in P), "Total_NPV_Benefit"\n        \n    # C1 & C2: Ràng buộc ngân sách\n    prob += pulp.lpSum(C[i] * y[i] for i in P) <= budget_total, "Budget_Total"\n    prob += pulp.lpSum(C1[i] * y[i] for i in P) <= budget_12, "Budget_12_Years"\n    \n    # C3: Ràng buộc loại trừ\n    if not force_p1_p2:\n        prob += y[1] + y[2] <= 1, "Exclusive_Datacenter"\n    else:\n        prob += y[1] == 1\n        prob += y[2] == 1\n        \n    # C4 & C5: Ràng buộc tiên quyết (phải đào tạo nhân lực y12 trước khi triển khai AI y8 và bán dẫn y13)\n    prob += y[8] <= y[12], "Prereq_AI_Training"\n    prob += y[13] <= y[12], "Prereq_Semi_Training"\n    \n    # C6: Ràng buộc cân đối lĩnh vực (Chính phủ số tối thiểu 1 dự án và An ninh mạng là bắt buộc)\n    prob += y[4] + y[5] >= 1, "Min_One_Gov"\n    prob += y[14] == 1, "Mandatory_CyberSecurity"\n    \n    # C7: Số lượng dự án chọn giới hạn từ 7 đến 11\n    prob += pulp.lpSum(y[i] for i in P) >= 7, "Min_Projects"\n    prob += pulp.lpSum(y[i] for i in P) <= 11, "Max_Projects"\n    \n    # Giải MIP bằng solver CBC mặc định\n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    # Trả về kết quả nếu tìm được lời giải khả thi\n    if prob.status != pulp.LpStatusOptimal:\n        # Nếu không khả thi và chúng ta bị ép buộc, trả về rỗng\n        return [], 0.0, 0.0\n        \n    selected = [i for i in P if y[i].varValue > 0.5]\n    total_cost = sum(C[i] for i in selected)\n    total_benefit = pulp.value(prob.objective)\n    \n    return selected, total_cost, total_benefit\n'
m4_labor_py_data = 'import numpy as np\nimport pandas as pd\nimport pulp\nfrom typing import Tuple, Dict, Any, List\n\nclass LaborMarketModel:\n    """\n    Lớp định nghĩa mô hình thị trường lao động Việt Nam dưới tác động của AI & Số hóa.\n    Bao gồm 8 ngành kinh tế chính với các tham số tương ứng.\n    """\n    def __init__(self):\n        self.sectors = {\n            1: "Nông-Lâm-Thủy sản",\n            2: "CN chế biến chế tạo",\n            3: "Xây dựng",\n            4: "Bán buôn-bán lẻ",\n            5: "Tài chính-Ngân hàng",\n            6: "Logistics-Vận tải",\n            7: "CNTT-Truyền thông",\n            8: "Giáo dục-Đào tạo"\n        }\n        \n        # Số lao động của từng ngành (triệu người)\n        self.L = np.array([13.20, 11.50, 4.80, 7.80, 0.55, 1.95, 0.62, 2.15])\n        \n        # Tỷ lệ rủi ro tự động hóa (%) của từng ngành\n        self.risk = np.array([18.0, 42.0, 25.0, 38.0, 52.0, 35.0, 28.0, 22.0]) / 100.0\n        \n        # Hệ số tạo việc làm mới, nâng cấp, dịch chuyển và đào tạo lại (số việc / tỷ VND đầu tư)\n        self.a1 = np.array([8.5, 32.5, 12.8, 22.4, 45.8, 28.5, 62.5, 18.5])\n        self.b1 = np.array([45.0, 28.0, 35.0, 32.0, 22.0, 30.0, 20.0, 55.0])\n        self.c1 = np.array([5.2, 62.4, 18.5, 48.2, 72.5, 42.8, 32.5, 12.5])\n        self.d1 = np.array([50.0, 32.0, 42.0, 38.0, 26.0, 36.0, 24.0, 62.0])\n\ndef solve_labor_optimization(budget: float = 30000.0) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:\n    """\n    Tối đa hóa tổng NetJob ròng trên toàn bộ 8 ngành dưới ngân sách đào tạo & AI cho trước.\n    \n    Args:\n        budget: Tổng ngân sách phân bổ cho cả 8 ngành (tỷ VND).\n        \n    Returns:\n        Tuple[np.ndarray, np.ndarray, np.ndarray, float]:\n            - x_AI_opt: Phân bổ đầu tư AI tối ưu từng ngành.\n            - x_H_opt: Phân bổ đầu tư đào tạo lại H tối ưu từng ngành.\n            - net_jobs_by_sector: Số việc làm ròng tăng thêm của mỗi ngành.\n            - max_net_jobs: Tổng số việc làm ròng cực đại.\n    """\n    model = LaborMarketModel()\n    N = len(model.sectors)\n    \n    prob = pulp.LpProblem("VN_Labor_Optimization", pulp.LpMaximize)\n    \n    # Biến quyết định\n    x_AI = pulp.LpVariable.dicts("x_AI", range(N), lowBound=0)\n    x_H = pulp.LpVariable.dicts("x_H", range(N), lowBound=0)\n    \n    # Định nghĩa các biểu thức trung gian cho mỗi ngành i\n    new_jobs = [model.a1[i] * x_AI[i] for i in range(N)]\n    upgrade_jobs = [model.b1[i] * x_H[i] for i in range(N)]\n    displaced_jobs = [model.c1[i] * model.risk[i] * x_AI[i] for i in range(N)]\n    retrain_caps = [model.d1[i] * x_H[i] for i in range(N)]\n    \n    # Hàm mục tiêu: max sum(NetJob_i)\n    prob += pulp.lpSum(new_jobs[i] + upgrade_jobs[i] - displaced_jobs[i] for i in range(N))\n    \n    # Ràng buộc\n    # 1. Ràng buộc ngân sách tổng\n    prob += pulp.lpSum(x_AI[i] + x_H[i] for i in range(N)) <= budget, "Total_Budget"\n    \n    # 2. NetJob_i >= 0 và Displaced_i <= RetrainCap_i cho mỗi ngành\n    for i in range(N):\n        prob += new_jobs[i] + upgrade_jobs[i] - displaced_jobs[i] >= 0, f"NetJob_NonNegative_Sector_{i+1}"\n        prob += displaced_jobs[i] <= retrain_caps[i], f"Retrain_Capacity_Sector_{i+1}"\n        \n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        raise ValueError("Không giải được bài toán tối ưu hóa thị trường lao động.")\n        \n    x_AI_opt = np.array([x_AI[i].varValue for i in range(N)])\n    x_H_opt = np.array([x_H[i].varValue for i in range(N)])\n    \n    # Tính toán số việc làm ròng cụ thể của mỗi ngành\n    net_jobs = np.array([\n        (model.a1[i] * x_AI_opt[i]) + (model.b1[i] * x_H_opt[i]) - (model.c1[i] * model.risk[i] * x_AI_opt[i])\n        for i in range(N)\n    ])\n    \n    return x_AI_opt, x_H_opt, net_jobs, pulp.value(prob.objective)\n\ndef minimum_training_threshold(sector_id: int = 2) -> float:\n    """\n    Tính toán ngưỡng đầu tư đào tạo lại tối thiểu (x_H) cho ngành Chế biến Chế tạo (sector_id = 2)\n    để đảm bảo NetJob >= 0 ngay cả khi phân bổ đầu tư AI tối đa.\n    Nguyên tắc: Tỷ lệ đầu tư đào tạo so với đầu tư AI phải đủ lớn để bù đắp rủi ro mất việc.\n    \n    Args:\n        sector_id: ID của ngành cần tính toán (1-indexed).\n        \n    Returns:\n        float: Tỷ trọng đầu tư x_H cần thiết tối thiểu trên mỗi đồng đầu tư x_AI (tỷ lệ x_H / x_AI).\n    """\n    model = LaborMarketModel()\n    idx = sector_id - 1 # Chuyển về 0-indexed\n    \n    a1_i = model.a1[idx]\n    b1_i = model.b1[idx]\n    c1_i = model.c1[idx]\n    risk_i = model.risk[idx]\n    \n    # NetJob_i = a1_i * x_AI_i + b1_i * x_H_i - c1_i * risk_i * x_AI_i >= 0\n    # -> x_H_i >= ((c1_i * risk_i - a1_i) / b1_i) * x_AI_i\n    val = (c1_i * risk_i - a1_i) / b1_i\n    return max(0.0, float(val))\n\ndef scenario_displacement_cap(\n    budget: float = 30000.0,\n    max_displacement_pct: float = 0.05\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float, bool]:\n    """\n    Mô phỏng thêm kịch bản ràng buộc bảo an xã hội: số lao động mất việc ở mỗi ngành\n    không vượt quá 5% tổng số lao động ban đầu của ngành đó (DisplacedJob_i <= 0.05 * L_i).\n    \n    Args:\n        budget: Tổng ngân sách.\n        max_displacement_pct: Tỷ lệ mất việc tối đa cho phép của ngành (ví dụ 0.05 = 5%).\n        \n    Returns:\n        Tuple[np.ndarray, np.ndarray, np.ndarray, float, bool]:\n            - x_AI_opt, x_H_opt: Phương án đầu tư.\n            - net_jobs: Số việc làm ròng từng ngành.\n            - max_net_jobs: Tổng số việc làm ròng.\n            - feasible: Trạng thái tìm được nghiệm khả thi (True/False).\n    """\n    model = LaborMarketModel()\n    N = len(model.sectors)\n    \n    prob = pulp.LpProblem("VN_Labor_Displacement_Cap", pulp.LpMaximize)\n    \n    x_AI = pulp.LpVariable.dicts("x_AI", range(N), lowBound=0)\n    x_H = pulp.LpVariable.dicts("x_H", range(N), lowBound=0)\n    \n    # Định nghĩa các biểu thức trung gian\n    new_jobs = [model.a1[i] * x_AI[i] for i in range(N)]\n    upgrade_jobs = [model.b1[i] * x_H[i] for i in range(N)]\n    displaced_jobs = [model.c1[i] * model.risk[i] * x_AI[i] for i in range(N)]\n    retrain_caps = [model.d1[i] * x_H[i] for i in range(N)]\n    \n    # Hàm mục tiêu\n    prob += pulp.lpSum(new_jobs[i] + upgrade_jobs[i] - displaced_jobs[i] for i in range(N))\n    \n    # Ràng buộc\n    prob += pulp.lpSum(x_AI[i] + x_H[i] for i in range(N)) <= budget\n    \n    for i in range(N):\n        prob += new_jobs[i] + upgrade_jobs[i] - displaced_jobs[i] >= 0\n        prob += displaced_jobs[i] <= retrain_caps[i]\n        # Thêm ràng buộc bảo an xã hội (Đổi đơn vị triệu người sang số người nếu cần,\n        # nhưng ở đây cả lao động và hệ số việc làm đều đồng bộ ở quy mô triệu người / việc làm)\n        # Vì L_i đo bằng triệu người, còn x_AI đo bằng tỷ đồng, hệ số việc làm đo bằng việc/tỷ đồng.\n        # Lưu ý: L_i triệu người = L_i * 1e6 người. Việc làm tạo ra = việc/tỷ VND * x_AI tỷ = số việc làm.\n        # Do đó, DisplacedJob_i (số việc làm bị thay thế) <= max_displacement_pct * (L_i * 1,000,000)\n        prob += displaced_jobs[i] <= max_displacement_pct * (model.L[i] * 1000000.0)\n        \n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        return np.zeros(N), np.zeros(N), np.zeros(N), 0.0, False\n        \n    x_AI_opt = np.array([x_AI[i].varValue for i in range(N)])\n    x_H_opt = np.array([x_H[i].varValue for i in range(N)])\n    \n    net_jobs = np.array([\n        (model.a1[i] * x_AI_opt[i]) + (model.b1[i] * x_H_opt[i]) - (model.c1[i] * model.risk[i] * x_AI_opt[i])\n        for i in range(N)\n    ])\n    \n    return x_AI_opt, x_H_opt, net_jobs, pulp.value(prob.objective), True\n\ndef get_sankey_data(x_AI_opt: np.ndarray, x_H_opt: np.ndarray) -> Dict[str, Any]:\n    """\n    Tạo dữ liệu cấu trúc cho biểu đồ Sankey thể hiện luồng chuyển dịch lao động\n    của các ngành dễ bị tổn thương (1 = Nông-Lâm-Thủy sản, 3 = Xây dựng, 4 = Bán buôn-bán lẻ).\n    \n    Returns:\n        Dict[str, Any]: Từ điển chứa các danh sách labels, sources, targets, values cho Sankey.\n    """\n    model = LaborMarketModel()\n    vulnerable_indices = [0, 2, 3] # Ngành 1, 3, 4 (0-indexed)\n    \n    labels = []\n    sources = []\n    targets = []\n    values = []\n    \n    # Các nút:\n    # 0, 1, 2: Lao động bị ảnh hưởng bởi tự động hóa (Ngành 1, 3, 4)\n    # 3, 4, 5: Lao động được đào tạo lại thành công (Upgrade/Retrained)\n    # 6: Lao động thất nghiệp lâm thời (Unemployed)\n    # 7: Thị trường việc làm mới từ AI (New Jobs)\n    \n    node_labels = [\n        "LĐ bị thay thế (Nông-Lâm-Thủy sản)",\n        "LĐ bị thay thế (Xây dựng)",\n        "LĐ bị thay thế (Bán buôn-bán lẻ)",\n        "LĐ đào tạo lại (Nông nghiệp)",\n        "LĐ đào tạo lại (Xây dựng)",\n        "LĐ đào tạo lại (Bán buôn-bán lẻ)",\n        "Lao động Thất nghiệp",\n        "Việc làm mới phát sinh từ AI"\n    ]\n    \n    for local_idx, idx in enumerate(vulnerable_indices):\n        # Tính toán chi tiết lao động của ngành idx\n        displaced = model.c1[idx] * model.risk[idx] * x_AI_opt[idx]\n        retrain_cap = model.d1[idx] * x_H_opt[idx]\n        \n        # Lao động đào tạo thành công tối đa bằng năng lực đào tạo hoặc phần bị thay thế\n        retrained = min(displaced, retrain_cap)\n        unemployed = max(0.0, displaced - retrained)\n        \n        # Việc làm mới từ AI trong ngành\n        new_job_ai = model.a1[idx] * x_AI_opt[idx]\n        \n        # Thêm luồng từ "LĐ bị thay thế" -> "LĐ đào tạo lại"\n        if retrained > 0:\n            sources.append(local_idx)\n            targets.append(3 + local_idx)\n            values.append(retrained)\n            \n        # Thêm luồng từ "LĐ bị thay thế" -> "Lao động Thất nghiệp"\n        if unemployed > 0:\n            sources.append(local_idx)\n            targets.append(6)\n            values.append(unemployed)\n            \n        # Thêm luồng từ "LĐ đào tạo lại" -> "Việc làm mới phát sinh từ AI"\n        if retrained > 0:\n            sources.append(3 + local_idx)\n            targets.append(7)\n            values.append(retrained)\n            \n    return {\n        \'labels\': node_labels,\n        \'sources\': sources,\n        \'targets\': targets,\n        \'values\': values\n    }\n\n# ==============================================================================\n# BỔ SUNG CÁC HÀM CHO 10 NGÀNH KINH TẾ (Notebooks, Tests & Dashboard)\n# ==============================================================================\n\ndef simulate_labor_dynamics(df_sectors: pd.DataFrame, x_inv: np.ndarray) -> pd.DataFrame:\n    """\n    Mô phỏng luồng biến động lao động của 10 ngành kinh tế dưới tác động của đầu tư AI/Số hóa.\n    \n    Args:\n        df_sectors: DataFrame chứa thông tin 10 ngành từ load_sectors().\n        x_inv: Mức đầu tư phân bổ cho mỗi ngành (billion VND).\n        \n    Returns:\n        pd.DataFrame: DataFrame bổ sung các cột jobs_created_thousand, jobs_displaced_thousand, net_jobs_thousand.\n    """\n    df = df_sectors.copy()\n    n_sectors = len(df)\n    \n    # Đảm bảo x_inv có cùng chiều dài với số ngành\n    if len(x_inv) < n_sectors:\n        x_inv_full = np.zeros(n_sectors)\n        x_inv_full[:len(x_inv)] = x_inv\n        x_inv = x_inv_full\n        \n    jobs_created = []\n    jobs_displaced = []\n    \n    for i in range(n_sectors):\n        row = df.iloc[i]\n        x_i = x_inv[i]\n        \n        # Hệ số tạo việc làm mới: phụ thuộc vào ai_readiness\n        ar_i = row.get(\'ai_readiness_0_100\', 50.0)\n        c_create = 0.12 * (ar_i / 100.0)\n        jobs_c = x_i * c_create\n        \n        # Hệ số sa thải/thay thế: phụ thuộc vào automation_risk và labor_million\n        risk_i = row.get(\'automation_risk_pct\', 30.0)\n        l_i = row.get(\'labor_million\', 2.0)\n        c_displace = 0.005 * (risk_i / 100.0) * l_i\n        jobs_d = x_i * c_displace\n        \n        jobs_created.append(jobs_c)\n        jobs_displaced.append(jobs_d)\n        \n    df[\'jobs_created_thousand\'] = jobs_created\n    df[\'jobs_displaced_thousand\'] = jobs_displaced\n    df[\'net_jobs_thousand\'] = df[\'jobs_created_thousand\'] - df[\'jobs_displaced_thousand\']\n    \n    return df\n\ndef solve_netjob_maximization(budget_total: float = 60000.0) -> Tuple[np.ndarray, float]:\n    """\n    Tối đa hóa tổng tạo việc làm ròng toàn nền kinh tế (cho 10 ngành) dưới ràng buộc ngân sách.\n    \n    Args:\n        budget_total: Tổng ngân sách đầu tư công nghệ & lao động (tỷ VND).\n        \n    Returns:\n        Tuple[np.ndarray, float]:\n            - x_opt: Phân bổ đầu tư tối ưu cho 10 ngành kinh tế.\n            - Z_opt: Tổng việc làm ròng cực đại (nghìn người).\n    """\n    from aideom_vn.src.data_loader import load_sectors\n    df_sectors = load_sectors()\n    n_sectors = len(df_sectors)\n    \n    prob = pulp.LpProblem("NetJob_Maximization", pulp.LpMaximize)\n    \n    # Biến quyết định: Phân bổ ngân sách cho từng ngành\n    x = [pulp.LpVariable(f\'x_netjob_{i}\', lowBound=1000.0, upBound=20000.0) for i in range(n_sectors)]\n    \n    # Hàm mục tiêu: Max maximize sum(NetJob_i)\n    net_jobs = []\n    for i in range(n_sectors):\n        row = df_sectors.iloc[i]\n        ar_i = row.get(\'ai_readiness_0_100\', 50.0)\n        risk_i = row.get(\'automation_risk_pct\', 30.0)\n        l_i = row.get(\'labor_million\', 2.0)\n        \n        c_create = 0.12 * (ar_i / 100.0)\n        c_displace = 0.005 * (risk_i / 100.0) * l_i\n        \n        net_jobs.append(x[i] * (c_create - c_displace))\n        \n    prob += pulp.lpSum(net_jobs)\n    \n    # Ràng buộc ngân sách tổng\n    prob += pulp.lpSum(x) == budget_total, "Budget_Constraint"\n    \n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        # Fallback chia đều ngân sách\n        x_opt = np.ones(n_sectors) * (budget_total / n_sectors)\n        Z_opt = 0.0\n        return x_opt, Z_opt\n        \n    x_opt = np.array([x[i].varValue for i in range(n_sectors)])\n    Z_opt = pulp.value(prob.objective)\n    \n    return x_opt, Z_opt\n\ndef get_minimum_retraining_threshold(\n    x_inv: float,\n    alpha_J: float = 0.12,\n    beta_J: float = 0.08,\n    k_displace: float = 0.005,\n    omega: float = 0.003,\n    c_re: float = 1.5\n) -> float:\n    """\n    Tính toán ngân sách đào tạo lại tối thiểu (x_H) cần thiết cho ngành Chế biến Chế tạo (sector_id = 2)\n    để đảm bảo số việc làm ròng không bị âm.\n    """\n    from aideom_vn.src.data_loader import load_sectors\n    df_sectors = load_sectors()\n    mfg = df_sectors[df_sectors[\'sector_id\'] == 2].iloc[0]\n    R_i = mfg[\'automation_risk_pct\']\n    L_i = mfg[\'labor_million\']\n    \n    # Số việc làm bị thay thế\n    jobs_displaced = x_inv * (k_displace * R_i + omega * L_i) / c_re\n    # Số việc làm mới tạo ra\n    jobs_created = x_inv * alpha_J\n    \n    # net_jobs = jobs_created + beta_J * x_H - jobs_displaced >= 0\n    # => beta_J * x_H >= jobs_displaced - jobs_created\n    min_x_H = (jobs_displaced - jobs_created) / beta_J\n    return max(0.0, float(min_x_H))\n\n'
m5_risk_py_data = 'import numpy as np\nimport pulp\nfrom typing import Dict, Any, Tuple, List, Optional\n\n# Thiết lập các tham số mặc định của bài toán Tối ưu hóa ngẫu nhiên 2 giai đoạn (Bài 10)\nDEFAULT_CONFIG = {\n    \'scenarios\': [\'Good\', \'Base\', \'Bad\'],\n    \'probabilities\': [0.3, 0.5, 0.2],\n    \'categories\': [\'Infra\', \'AI\', \'Human\'],\n    \'first_stage_cost\': [8.0, 12.0, 10.0],  # c_i\n    \'budget\': 100.0,\n    \'min_investment\': [10.0, 10.0, 10.0],  # x_i >= 10\n    # Nhu cầu thị trường d_{s,i}\n    \'demands\': {\n        \'Good\': [50.0, 40.0, 45.0],\n        \'Base\': [30.0, 25.0, 30.0],\n        \'Bad\': [15.0, 10.0, 15.0]\n    },\n    # Doanh thu biên ở giai đoạn 2 q_{s,i}\n    \'revenues\': {\n        \'Good\': [15.0, 22.0, 18.0],\n        \'Base\': [12.0, 18.0, 14.0],\n        \'Bad\': [8.0, 10.0, 9.0]\n    }\n}\n\ndef solve_stochastic_rp_pulp(config: Dict[str, Any] = DEFAULT_CONFIG) -> Tuple[np.ndarray, Dict[str, np.ndarray], float]:\n    """\n    Giải Bài toán Lập trình Ngẫu nhiên với Quyết định bù đắp (Recourse Problem - RP) bằng PuLP.\n    Mục tiêu: Cực đại hóa Lợi nhuận Kỳ vọng ròng.\n    E[Net Profit] = sum_s p_s * (sum_i q_{s,i} * y_{s,i}) - sum_i c_i * x_i\n    \n    Returns:\n        Tuple[np.ndarray, Dict[str, np.ndarray], float]:\n            - x_opt: Quyết định giai đoạn 1 tối ưu (Đầu tư trước khi biết kịch bản).\n            - y_opt_s: Quyết định giai đoạn 2 cho từng kịch bản s.\n            - Z_RP: Lợi nhuận kỳ vọng ròng cực đại.\n    """\n    scenarios = config[\'scenarios\']\n    probs = config[\'probabilities\']\n    categories = config[\'categories\']\n    c = config[\'first_stage_cost\']\n    budget = config[\'budget\']\n    min_x = config[\'min_investment\']\n    demands = config[\'demands\']\n    revenues = config[\'revenues\']\n    \n    n_cats = len(categories)\n    \n    # Khởi tạo bài toán\n    prob = pulp.LpProblem("Stochastic_Recourse_Problem", pulp.LpMaximize)\n    \n    # Biến quyết định giai đoạn 1\n    x = [pulp.LpVariable(f\'x_{cat}\', lowBound=min_x[i]) for i, cat in enumerate(categories)]\n    \n    # Biến quyết định giai đoạn 2 (phụ thuộc vào từng kịch bản s)\n    y = {}\n    for s in scenarios:\n        y[s] = [pulp.LpVariable(f\'y_{s}_{cat}\', lowBound=0) for cat in categories]\n        \n    # Hàm mục tiêu: Lợi nhuận Kỳ vọng ròng\n    # Doanh thu giai đoạn 2 kỳ vọng\n    expected_revenue = pulp.lpSum(\n        probs[s_idx] * pulp.lpSum(revenues[s][i] * y[s][i] for i in range(n_cats))\n        for s_idx, s in enumerate(scenarios)\n    )\n    # Chi phí giai đoạn 1\n    first_stage_cost = pulp.lpSum(c[i] * x[i] for i in range(n_cats))\n    \n    prob += expected_revenue - first_stage_cost, "Expected_Net_Profit"\n    \n    # Ràng buộc giai đoạn 1: Ngân sách đầu tư\n    prob += pulp.lpSum(x[i] for i in range(n_cats)) <= budget, "Budget_Constraint"\n    \n    # Ràng buộc giai đoạn 2:\n    for s_idx, s in enumerate(scenarios):\n        for i in range(n_cats):\n            # y_{s,i} <= x_i (Quyết định vận hành không vượt quá năng lực đã đầu tư)\n            prob += y[s][i] <= x[i], f"Capacity_Limit_{s}_{categories[i]}"\n            # y_{s,i} <= d_{s,i} (Quyết định vận hành không vượt quá nhu cầu thị trường)\n            prob += y[s][i] <= demands[s][i], f"Demand_Limit_{s}_{categories[i]}"\n            \n    # Giải bài toán\n    prob.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    if prob.status != pulp.LpStatusOptimal:\n        raise ValueError("Không tìm thấy nghiệm tối ưu cho bài toán RP.")\n        \n    x_opt = np.array([x[i].varValue for i in range(n_cats)])\n    \n    y_opt_s = {}\n    for s in scenarios:\n        y_opt_s[s] = np.array([y[s][i].varValue for i in range(n_cats)])\n        \n    Z_RP = pulp.value(prob.objective)\n    \n    return x_opt, y_opt_s, Z_RP\n\ndef solve_expected_value_problem_pulp(config: Dict[str, Any] = DEFAULT_CONFIG) -> Tuple[np.ndarray, float, float]:\n    """\n    1. Giải bài toán Tất định Giá trị Kỳ vọng (Expected Value problem - EV) bằng cách\n       thay thế các yếu tố ngẫu nhiên bằng kỳ vọng toán học của chúng.\n    2. Sử dụng quyết định x_EV tìm được để tính EEV (Expected Result of Expected Value)\n       bằng cách cố định x = x_EV và giải bài toán tối ưu giai đoạn 2 trên từng kịch bản.\n       \n    Returns:\n        Tuple[np.ndarray, float, float]:\n            - x_EV: Quyết định giai đoạn 1 từ bài toán EV.\n            - Z_EV: Mục tiêu của bài toán EV (không phải EEV).\n            - Z_EEV: Lợi nhuận kỳ vọng thực tế khi áp dụng x_EV (EEV).\n    """\n    scenarios = config[\'scenarios\']\n    probs = config[\'probabilities\']\n    categories = config[\'categories\']\n    c = config[\'first_stage_cost\']\n    budget = config[\'budget\']\n    min_x = config[\'min_investment\']\n    demands = config[\'demands\']\n    revenues = config[\'revenues\']\n    \n    n_cats = len(categories)\n    \n    # ------------------ BƯỚC 1: GIẢI BÀI TOÁN EV (EXPECTED VALUE) ------------------\n    # Tính nhu cầu kỳ vọng và doanh thu kỳ vọng\n    mean_demand = np.zeros(n_cats)\n    mean_revenue = np.zeros(n_cats)\n    for s_idx, s in enumerate(scenarios):\n        mean_demand += probs[s_idx] * np.array(demands[s])\n        mean_revenue += probs[s_idx] * np.array(revenues[s])\n        \n    prob_ev = pulp.LpProblem("Expected_Value_Problem", pulp.LpMaximize)\n    \n    x_ev_vars = [pulp.LpVariable(f\'x_ev_{cat}\', lowBound=min_x[i]) for i, cat in enumerate(categories)]\n    y_ev_vars = [pulp.LpVariable(f\'y_ev_{cat}\', lowBound=0) for cat in categories]\n    \n    # Hàm mục tiêu bài toán EV\n    ev_revenue = pulp.lpSum(mean_revenue[i] * y_ev_vars[i] for i in range(n_cats))\n    ev_cost = pulp.lpSum(c[i] * x_ev_vars[i] for i in range(n_cats))\n    prob_ev += ev_revenue - ev_cost, "EV_Objective"\n    \n    # Ràng buộc\n    prob_ev += pulp.lpSum(x_ev_vars[i] for i in range(n_cats)) <= budget, "EV_Budget"\n    for i in range(n_cats):\n        prob_ev += y_ev_vars[i] <= x_ev_vars[i]\n        prob_ev += y_ev_vars[i] <= mean_demand[i]\n        \n    prob_ev.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    x_EV = np.array([x_ev_vars[i].varValue for i in range(n_cats)])\n    Z_EV = pulp.value(prob_ev.objective)\n    \n    # ------------------ BƯỚC 2: TÍNH EEV VỚI x CỐ ĐỊNH = x_EV ------------------\n    # Với mỗi kịch bản s, ta cố định x = x_EV và tìm y_s tối ưu\n    eev_revenues = []\n    \n    for s in scenarios:\n        prob_s = pulp.LpProblem(f"EEV_Scenario_{s}", pulp.LpMaximize)\n        y_s_vars = [pulp.LpVariable(f\'y_eev_{s}_{cat}\', lowBound=0) for cat in categories]\n        \n        prob_s += pulp.lpSum(revenues[s][i] * y_s_vars[i] for i in range(n_cats)), "Scenario_Revenue"\n        \n        for i in range(n_cats):\n            prob_s += y_s_vars[i] <= x_EV[i]\n            prob_s += y_s_vars[i] <= demands[s][i]\n            \n        prob_s.solve(pulp.PULP_CBC_CMD(msg=False))\n        eev_revenues.append(pulp.value(prob_s.objective))\n        \n    # Lợi nhuận kỳ vọng ròng EEV\n    first_stage_cost_ev = np.sum(c * x_EV)\n    expected_eev_revenue = np.sum(np.array(probs) * np.array(eev_revenues))\n    Z_EEV = expected_eev_revenue - first_stage_cost_ev\n    \n    return x_EV, Z_EV, Z_EEV\n\ndef solve_wait_and_see_pulp(config: Dict[str, Any] = DEFAULT_CONFIG) -> Tuple[Dict[str, np.ndarray], float]:\n    """\n    Giải bài toán Chờ và Xem (Wait-and-See - WS).\n    Giả định nhà hoạch định chính sách có thông tin hoàn hảo về kịch bản trước khi ra quyết định đầu tư x.\n    Tương đương giải bài toán tất định riêng lẻ cho từng kịch bản s rồi tính kỳ vọng.\n    \n    Returns:\n        Tuple[Dict[str, np.ndarray], float]:\n            - x_opt_s: Quyết định đầu tư tối ưu cho từng kịch bản s.\n            - Z_WS: Lợi nhuận kỳ vọng của Wait-and-See.\n    """\n    scenarios = config[\'scenarios\']\n    probs = config[\'probabilities\']\n    categories = config[\'categories\']\n    c = config[\'first_stage_cost\']\n    budget = config[\'budget\']\n    min_x = config[\'min_investment\']\n    demands = config[\'demands\']\n    revenues = config[\'revenues\']\n    \n    n_cats = len(categories)\n    \n    ws_profits = []\n    x_opt_s = {}\n    \n    for s_idx, s in enumerate(scenarios):\n        prob_s = pulp.LpProblem(f"WS_Scenario_{s}", pulp.LpMaximize)\n        \n        x_vars = [pulp.LpVariable(f\'x_ws_{s}_{cat}\', lowBound=min_x[i]) for i, cat in enumerate(categories)]\n        y_vars = [pulp.LpVariable(f\'y_ws_{s}_{cat}\', lowBound=0) for cat in categories]\n        \n        # Lợi nhuận ròng của kịch bản s\n        revenue = pulp.lpSum(revenues[s][i] * y_vars[i] for i in range(n_cats))\n        cost = pulp.lpSum(c[i] * x_vars[i] for i in range(n_cats))\n        prob_s += revenue - cost, "Scenario_Net_Profit"\n        \n        prob_s += pulp.lpSum(x_vars[i] for i in range(n_cats)) <= budget, "Budget_Constraint"\n        for i in range(n_cats):\n            prob_s += y_vars[i] <= x_vars[i]\n            prob_s += y_vars[i] <= demands[s][i]\n            \n        prob_s.solve(pulp.PULP_CBC_CMD(msg=False))\n        \n        ws_profits.append(pulp.value(prob_s.objective))\n        x_opt_s[s] = np.array([x_vars[i].varValue for i in range(n_cats)])\n        \n    Z_WS = np.sum(np.array(probs) * np.array(ws_profits))\n    \n    return x_opt_s, Z_WS\n\ndef solve_minimax_regret_pulp(config: Dict[str, Any] = DEFAULT_CONFIG) -> Tuple[np.ndarray, float]:\n    """\n    Giải bài toán Tối thiểu hóa Hối tiếc Cực đại (Robust Minimax Regret).\n    Mục tiêu: Lựa chọn x sao cho hối tiếc lớn nhất giữa lợi nhuận tối ưu của kịch bản s (Z*_s)\n    và lợi nhuận thực tế đạt được dưới quyết định x là nhỏ nhất.\n    Minimizes H, subject to:\n    H >= Z*_s - [sum_i q_{s,i} * y_{s,i} - sum_i c_i * x_i]  với mọi kịch bản s.\n    \n    Returns:\n        Tuple[np.ndarray, float]:\n            - x_opt: Quyết định đầu tư cực kỳ vững chắc (robust).\n            - max_regret: Giá trị hối tiếc cực đại tối thiểu hóa.\n    """\n    scenarios = config[\'scenarios\']\n    categories = config[\'categories\']\n    c = config[\'first_stage_cost\']\n    budget = config[\'budget\']\n    min_x = config[\'min_investment\']\n    demands = config[\'demands\']\n    revenues = config[\'revenues\']\n    \n    n_cats = len(categories)\n    \n    # Bước 1: Tính lợi nhuận tối ưu tuyệt đối Z*_s cho từng kịch bản s độc lập\n    x_opt_s, _ = solve_wait_and_see_pulp(config)\n    Z_star_s = {}\n    \n    for s in scenarios:\n        # Giải lại nhanh để lấy trị số mục tiêu chính xác của kịch bản s\n        prob_s = pulp.LpProblem(f"Z_star_{s}", pulp.LpMaximize)\n        x_vars = [pulp.LpVariable(f\'x_star_{s}_{cat}\', lowBound=min_x[i]) for i, cat in enumerate(categories)]\n        y_vars = [pulp.LpVariable(f\'y_star_{s}_{cat}\', lowBound=0) for cat in categories]\n        \n        prob_s += pulp.lpSum(revenues[s][i] * y_vars[i] for i in range(n_cats)) - pulp.lpSum(c[i] * x_vars[i] for i in range(n_cats))\n        prob_s += pulp.lpSum(x_vars[i] for i in range(n_cats)) <= budget\n        for i in range(n_cats):\n            prob_s += y_vars[i] <= x_vars[i]\n            prob_s += y_vars[i] <= demands[s][i]\n            \n        prob_s.solve(pulp.PULP_CBC_CMD(msg=False))\n        Z_star_s[s] = pulp.value(prob_s.objective)\n        \n    # Bước 2: Thiết lập bài toán tối thiểu hóa hối tiếc cực đại\n    prob_regret = pulp.LpProblem("Minimax_Regret_Problem", pulp.LpMinimize)\n    \n    # Biến quyết định giai đoạn 1\n    x = [pulp.LpVariable(f\'x_regret_{cat}\', lowBound=min_x[i]) for i, cat in enumerate(categories)]\n    \n    # Biến quyết định giai đoạn 2 cho từng kịch bản\n    y = {}\n    for s in scenarios:\n        y[s] = [pulp.LpVariable(f\'y_regret_{s}_{cat}\', lowBound=0) for cat in categories]\n        \n    # Biến phụ biểu thị hối tiếc cực đại H\n    H = pulp.LpVariable("Max_Regret", lowBound=0)\n    \n    prob_regret += H, "Minimize_Max_Regret"\n    \n    # Ràng buộc ngân sách giai đoạn 1\n    prob_regret += pulp.lpSum(x[i] for i in range(n_cats)) <= budget, "Budget_Constraint"\n    \n    first_stage_cost = pulp.lpSum(c[i] * x[i] for i in range(n_cats))\n    \n    # Ràng buộc giai đoạn 2 và Ràng buộc Hối tiếc cho từng kịch bản\n    for s in scenarios:\n        for i in range(n_cats):\n            prob_regret += y[s][i] <= x[i], f"Cap_Limit_{s}_{categories[i]}"\n            prob_regret += y[s][i] <= demands[s][i], f"Dem_Limit_{s}_{categories[i]}"\n            \n        # Regret_s = Z*_s - (Revenue_s - Cost_1st)\n        profit_s = pulp.lpSum(revenues[s][i] * y[s][i] for i in range(n_cats)) - first_stage_cost\n        prob_regret += H >= Z_star_s[s] - profit_s, f"Regret_Constraint_{s}"\n        \n    prob_regret.solve(pulp.PULP_CBC_CMD(msg=False))\n    \n    x_opt = np.array([x[i].varValue for i in range(n_cats)])\n    max_regret = pulp.value(H)\n    \n    return x_opt, max_regret\n\ndef calculate_risk_metrics(config: Dict[str, Any] = DEFAULT_CONFIG) -> Dict[str, Any]:\n    """\n    Tính toán toàn bộ các chỉ số đo lường rủi ro kinh tế:\n    - RP (Recourse Problem)\n    - EV (Expected Value Problem)\n    - EEV (Expected result of Expected Value)\n    - WS (Wait-and-See)\n    - VSS (Value of Stochastic Solution) = RP - EEV\n    - EVPI (Expected Value of Perfect Information) = WS - RP\n    \n    Returns:\n        Dict[str, Any]: Từ điển chứa các kết quả tính toán chi tiết.\n    """\n    x_RP, _, Z_RP = solve_stochastic_rp_pulp(config)\n    x_EV, Z_EV, Z_EEV = solve_expected_value_problem_pulp(config)\n    _, Z_WS = solve_wait_and_see_pulp(config)\n    \n    VSS = Z_RP - Z_EEV\n    EVPI = Z_WS - Z_RP\n    \n    return {\n        \'x_RP\': x_RP,\n        \'Z_RP\': Z_RP,\n        \'x_EV\': x_EV,\n        \'Z_EV\': Z_EV,\n        \'Z_EEV\': Z_EEV,\n        \'Z_WS\': Z_WS,\n        \'VSS\': VSS,\n        \'EVPI\': EVPI\n    }\n\n# ----------------- PHẦN PYOMO HOÀN THIỆN (CÓ CHẾ CHẾ FALLBACK SANG PULP) -----------------\ntry:\n    import pyomo.environ as pyo\n    PYOMO_AVAILABLE = True\nexcept ImportError:\n    PYOMO_AVAILABLE = False\n\ndef solve_stochastic_pyomo(config: Dict[str, Any] = DEFAULT_CONFIG) -> Tuple[np.ndarray, float]:\n    """\n    Giải bài toán lập trình ngẫu nhiên bằng Pyomo.\n    Nếu hệ thống không có sẵn solver thích hợp (như glpk hoặc cbc) được cài ngoài,\n    hàm sẽ tự động chuyển hướng và giải bằng PuLP để đảm bảo hệ thống không bị crash.\n    """\n    if not PYOMO_AVAILABLE:\n        # Fallback sang PuLP nếu không import được pyomo\n        x_opt, _, Z_RP = solve_stochastic_rp_pulp(config)\n        return x_opt, Z_RP\n        \n    scenarios = config[\'scenarios\']\n    probs = config[\'probabilities\']\n    categories = config[\'categories\']\n    c = config[\'first_stage_cost\']\n    budget = config[\'budget\']\n    min_x = config[\'min_investment\']\n    demands = config[\'demands\']\n    revenues = config[\'revenues\']\n    \n    n_cats = len(categories)\n    \n    model = pyo.ConcreteModel()\n    \n    # Khai báo tập chỉ số\n    model.I = pyo.RangeSet(0, n_cats - 1)\n    model.S = pyo.Set(initialize=scenarios)\n    \n    # Biến quyết định giai đoạn 1\n    def x_bounds(model, i):\n        return (min_x[i], None)\n    model.x = pyo.Var(model.I, bounds=x_bounds)\n    \n    # Biến quyết định giai đoạn 2\n    model.y = pyo.Var(model.S, model.I, bounds=(0, None))\n    \n    # Hàm mục tiêu: Kỳ vọng lợi nhuận ròng\n    def obj_rule(model):\n        exp_revenue = sum(\n            probs[s_idx] * sum(revenues[s][i] * model.y[s, i] for i in model.I)\n            for s_idx, s in enumerate(scenarios)\n        )\n        cost = sum(c[i] * model.x[i] for i in model.I)\n        return exp_revenue - cost\n    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)\n    \n    # Ràng buộc ngân sách giai đoạn 1\n    def budget_rule(model):\n        return sum(model.x[i] for i in model.I) <= budget\n    model.budget_constraint = pyo.Constraint(rule=budget_rule)\n    \n    # Ràng buộc giai đoạn 2 cho biến y\n    def cap_rule(model, s, i):\n        return model.y[s, i] <= model.x[i]\n    model.cap_constraints = pyo.Constraint(model.S, model.I, rule=cap_rule)\n    \n    def demand_rule(model, s, i):\n        return model.y[s, i] <= demands[s][i]\n    model.demand_constraints = pyo.Constraint(model.S, model.I, rule=demand_rule)\n    \n    # Thử giải bằng solver glpk hoặc cbc qua Pyomo\n    # Nếu thất bại, tự động fallback sang PuLP\n    solved = False\n    for solver_name in [\'glpk\', \'cbc\']:\n        try:\n            solver = pyo.SolverFactory(solver_name)\n            results = solver.solve(model, tee=False)\n            if (results.solver.status == pyo.SolverStatus.ok) and (results.solver.termination_condition == pyo.TerminationCondition.optimal):\n                solved = True\n                break\n        except Exception:\n            continue\n            \n    if solved:\n        x_opt = np.array([pyo.value(model.x[i]) for i in model.I])\n        Z_RP = pyo.value(model.obj)\n        return x_opt, Z_RP\n    else:\n        # Fallback sang PuLP solver\n        x_opt, _, Z_RP = solve_stochastic_rp_pulp(config)\n        return x_opt, Z_RP\n'
rl_env_py_data = 'import numpy as np\nimport gymnasium as gym\nfrom gymnasium import spaces\nfrom typing import Tuple, Dict, Any, Optional\n\nclass VietnamEconomyEnv(gym.Env):\n    """\n    Môi trường Gymnasium mô phỏng kinh tế Việt Nam trong kỷ nguyên AI (Bài 11).\n    Không gian trạng thái gồm 81 trạng thái rời rạc đại diện cho sự kết hợp của:\n    1. Trình độ sẵn sàng AI (AI Readiness): 0 (Thấp), 1 (Trung bình), 2 (Cao) -> 3 trạng thái\n    2. Tốc độ tăng trưởng GDP (GDP Growth): 0 (Chậm), 1 (Trung bình), 2 (Nhanh) -> 3 trạng thái\n    3. Rủi ro thất nghiệp công nghệ (Labor Risk): 0 (Thấp), 1 (Trung bình), 2 (Cao) -> 3 trạng thái\n    4. Tỷ lệ lao động qua đào tạo (Trained Labor): 0 (Thấp), 1 (Trung bình), 2 (Cao) -> 3 trạng thái\n    Tổng số trạng thái = 3 * 3 * 3 * 3 = 81.\n    \n    Không gian hành động gồm 5 chính sách vĩ mô (Actions):\n    - Action 0: Tập trung đầu tư hạ tầng vật chất (Infra Focus)\n    - Action 1: Đẩy mạnh phát triển AI & Công nghệ số (AI & Digital Focus)\n    - Action 2: Ưu tiên đào tạo nhân lực số (Human Capital Focus)\n    - Action 3: Phát triển cân bằng (Balanced Investment)\n    - Action 4: Tăng cường an sinh xã hội & đào tạo lại (Social Safety Net & Retraining)\n    """\n    metadata = {"render_modes": ["human"]}\n    \n    def __init__(self):\n        super(VietnamEconomyEnv, self).__init__()\n        \n        # 5 hành động rời rạc\n        self.action_space = spaces.Discrete(5)\n        \n        # 81 trạng thái rời rạc\n        self.observation_space = spaces.Discrete(81)\n        \n        # Khởi tạo trạng thái ban đầu: Trình độ trung bình (AI=1, GDP=1, Risk=1, Train=1)\n        # Cách giải mã trạng thái state -> (ai, gdp, risk, train):\n        # state = ai * 27 + gdp * 9 + risk * 3 + train\n        self.state = 1 * 27 + 1 * 9 + 1 * 3 + 1\n        self.steps = 0\n        self.max_steps = 20  # Một đợt hoạch định chính sách kéo dài 20 năm (hoặc 20 bước)\n        \n    def _decode_state(self, state: int) -> Tuple[int, int, int, int]:\n        ai = state // 27\n        rem = state % 27\n        gdp = rem // 9\n        rem = rem % 9\n        risk = rem // 3\n        train = rem % 3\n        return ai, gdp, risk, train\n        \n    def _encode_state(self, ai: int, gdp: int, risk: int, train: int) -> int:\n        # Giới hạn các giá trị trong khoảng [0, 2]\n        ai = int(np.clip(ai, 0, 2))\n        gdp = int(np.clip(gdp, 0, 2))\n        risk = int(np.clip(risk, 0, 2))\n        train = int(np.clip(train, 0, 2))\n        return ai * 27 + gdp * 9 + risk * 3 + train\n        \n    def reset(self, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None) -> Tuple[int, Dict[str, Any]]:\n        super().reset(seed=seed)\n        self.steps = 0\n        # Bắt đầu tại trạng thái ổn định cơ sở (AI=1, GDP=1, Risk=1, Trained=1) -> state = 40\n        self.state = self._encode_state(1, 1, 1, 1)\n        return self.state, {}\n        \n    def step(self, action: int) -> Tuple[int, float, bool, bool, Dict[str, Any]]:\n        self.steps += 1\n        ai, gdp, risk, train = self._decode_state(self.state)\n        \n        # Lưu các giá trị cũ để phân tích\n        prev_ai, prev_gdp, prev_risk, prev_train = ai, gdp, risk, train\n        \n        # Quy tắc chuyển đổi trạng thái mang tính kinh tế vĩ mô\n        if action == 0:  # Tập trung hạ tầng\n            gdp = gdp + 1 if np.random.rand() < 0.7 else gdp\n            # Đầu tư hạ tầng truyền thống không giúp tăng trình độ AI hay nhân lực nhiều\n            ai = ai - 1 if np.random.rand() < 0.2 else ai\n            \n        elif action == 1:  # Tập trung công nghệ AI & số hóa\n            ai = ai + 1 if np.random.rand() < 0.8 else ai\n            gdp = gdp + 1 if np.random.rand() < 0.5 else gdp\n            # AI phát triển nhanh làm tăng rủi ro thất nghiệp nếu không đào tạo kịp thời\n            if train < 2:\n                risk = risk + 1 if np.random.rand() < 0.8 else risk\n            else:\n                risk = risk + 1 if np.random.rand() < 0.3 else risk\n                \n        elif action == 2:  # Ưu tiên nhân lực đào tạo\n            train = train + 1 if np.random.rand() < 0.8 else train\n            # Nhân lực chất lượng cao giúp giảm rủi ro thất nghiệp và tăng khả năng sẵn sàng AI\n            risk = risk - 1 if np.random.rand() < 0.6 else risk\n            ai = ai + 1 if np.random.rand() < 0.4 else ai\n            \n        elif action == 3:  # Phát triển cân bằng\n            # Tăng nhẹ ở nhiều khía cạnh\n            if np.random.rand() < 0.5:\n                gdp = min(2, gdp + 1)\n            if np.random.rand() < 0.4:\n                ai = min(2, ai + 1)\n            if np.random.rand() < 0.4:\n                train = min(2, train + 1)\n            # Rủi ro thất nghiệp được kiềm chế\n            risk = risk - 1 if np.random.rand() < 0.3 else risk\n            \n        elif action == 4:  # An sinh xã hội & Đào tạo lại\n            # Giảm rủi ro thất nghiệp mạnh mẽ và tăng nhẹ trình độ nhân lực\n            risk = risk - 1 if np.random.rand() < 0.9 else risk\n            train = train + 1 if np.random.rand() < 0.5 else train\n            # Giảm nhẹ tài lực đầu tư phát triển gdp ngắn hạn\n            if np.random.rand() < 0.1:\n                gdp = max(0, gdp - 1)\n                \n        # Giới hạn trạng thái và cập nhật trạng thái mới\n        self.state = self._encode_state(ai, gdp, risk, train)\n        \n        # --- HÀM THƯỞNG PHÚC LỢI KINH TẾ (REWARD FUNCTION) ---\n        # GDP tăng trưởng nhanh (gdp=2) đem lại điểm cộng lớn\n        gdp_reward = [0.0, 10.0, 25.0][gdp]\n        \n        # Mức độ sẵn sàng AI đem lại hiệu suất kinh tế dài hạn\n        ai_reward = [0.0, 5.0, 15.0][ai]\n        \n        # Nhân lực trình độ cao giúp gia tăng năng suất bền vững\n        train_reward = [0.0, 5.0, 12.0][train]\n        \n        # Thất nghiệp cao (risk=2) chịu hình phạt rất nặng (phúc lợi xã hội giảm sâu)\n        risk_penalty = [0.0, -8.0, -30.0][risk]\n        \n        # Chi phí của các hành động chính sách (cost of intervention)\n        action_cost = {0: -4.0, 1: -6.0, 2: -5.0, 3: -7.0, 4: -3.0}[action]\n        \n        # Phạt nếu để xảy ra khủng hoảng kép (GDP kém và Thất nghiệp cao)\n        crisis_penalty = -20.0 if (gdp == 0 and risk == 2) else 0.0\n        \n        reward = gdp_reward + ai_reward + train_reward + risk_penalty + action_cost + crisis_penalty\n        \n        # Kiểm tra trạng thái kết thúc\n        terminated = self.steps >= self.max_steps\n        truncated = False\n        \n        info = {\n            \'ai_level\': ai,\n            \'gdp_growth\': gdp,\n            \'labor_risk\': risk,\n            \'trained_labor\': train,\n            \'step_index\': self.steps\n        }\n        \n        return self.state, float(reward), terminated, truncated, info\n\ndef train_q_learning(\n    env: VietnamEconomyEnv,\n    episodes: int = 2000,\n    alpha: float = 0.1,      # Học suất (learning rate)\n    gamma: float = 0.9,      # Hệ số chiết khấu (discount factor)\n    epsilon: float = 1.0,    # Tham số khám phá ban đầu\n    min_epsilon: float = 0.05,\n    decay_rate: float = 0.995\n) -> Tuple[np.ndarray, list]:\n    """\n    Huấn luyện Agent bằng thuật toán Q-learning rời rạc trên môi trường VietnamEconomyEnv.\n    \n    Returns:\n        Tuple[np.ndarray, list]:\n            - Q_table: Ma trận Q-value kích thước (81, 5).\n            - rewards_history: Lịch sử tổng phần thưởng qua các tập huấn luyện.\n    """\n    n_states = env.observation_space.n\n    n_actions = env.action_space.n\n    \n    # Khởi tạo Q-table bằng 0\n    Q_table = np.zeros((n_states, n_actions))\n    rewards_history = []\n    \n    for episode in range(episodes):\n        state, _ = env.reset()\n        total_reward = 0\n        done = False\n        \n        while not done:\n            # Chọn hành động bằng chiến lược Epsilon-Greedy\n            if np.random.rand() < epsilon:\n                action = env.action_space.sample()  # Khám phá (Exploration)\n            else:\n                action = np.argmax(Q_table[state, :])  # Khai thác (Exploitation)\n                \n            # Thực thi bước hành động\n            next_state, reward, terminated, truncated, _ = env.step(action)\n            done = terminated or truncated\n            \n            # Cập nhật giá trị Q bằng phương trình Bellman sửa đổi\n            best_next_action = np.argmax(Q_table[next_state, :])\n            td_target = reward + gamma * Q_table[next_state, best_next_action]\n            td_error = td_target - Q_table[state, action]\n            Q_table[state, action] += alpha * td_error\n            \n            state = next_state\n            total_reward += reward\n            \n        # Giảm dần mức độ khám phá epsilon\n        epsilon = max(min_epsilon, epsilon * decay_rate)\n        rewards_history.append(total_reward)\n        \n    return Q_table, rewards_history\n'

with open("aideom_vn/src/__init__.py", "w", encoding="utf-8") as f:
    f.write("")

with open("aideom_vn/src/data_loader.py", "w", encoding="utf-8") as f:
    f.write(data_loader_py_data)

with open("aideom_vn/src/m1_production.py", "w", encoding="utf-8") as f:
    f.write(m1_production_py_data)

with open("aideom_vn/src/m2_readiness.py", "w", encoding="utf-8") as f:
    f.write(m2_readiness_py_data)

with open("aideom_vn/src/m3_allocation.py", "w", encoding="utf-8") as f:
    f.write(m3_allocation_py_data)

with open("aideom_vn/src/m4_labor.py", "w", encoding="utf-8") as f:
    f.write(m4_labor_py_data)

with open("aideom_vn/src/m5_risk.py", "w", encoding="utf-8") as f:
    f.write(m5_risk_py_data)

with open("aideom_vn/src/rl_env.py", "w", encoding="utf-8") as f:
    f.write(rl_env_py_data)

# 5. Cấu hình PYTHONPATH để import bình thường
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

print("Môi trường đã được cấu hình hoàn tất! Sẵn sàng bắt đầu bài học.")



## 1. Tối Ưu Hóa Đa Mục Tiêu Bằng Thuật Toán Tiến Hóa Pareto (NSGA-II tinh giản)
Chúng ta tìm kiếm tập hợp các phương án phân bổ ngân sách tối ưu Pareto để giải quyết xung đột giữa hai mục tiêu chính:
1. **Mục tiêu 1 (Tăng trưởng)**: Cực đại hóa GDP tăng thêm quốc gia ($f_1 = 0.85x_1 + 1.20x_2 + 0.95x_3 + 1.35x_4$).
2. **Mục tiêu 2 (Công bằng/Bình đẳng)**: Cực tiểu hóa sự chênh lệch phân bổ giữa các vùng miền hoặc cực đại hóa phúc lợi xã hội (ở đây mô phỏng bằng việc phân bổ hài hòa và hạn chế tập trung quá mức).

Ta sẽ lập trình một giải thuật tìm kiếm Pareto bằng phương pháp Weighted-Sum quét qua các trọng số của mục tiêu hoặc một thuật toán tìm kiếm ngẫu nhiên có sàng lọc Pareto rất trực quan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

# Định nghĩa bài toán đa mục tiêu phân bổ ngân sách
# Ta giải bài toán bằng phương pháp tổng trọng số (Weighted Sum Method) với w1*F1 + w2*F2
# để tạo ra tập hợp biên Pareto (Pareto Frontier) cực kỳ chuẩn xác.
def solve_weighted_multiobjective(w1: float, budget: float = 100.0) -> Tuple[np.ndarray, float, float]:
    w2 = 1.0 - w1
    prob = pulp.LpProblem("MultiObjective_Budget", pulp.LpMaximize)
    
    x1 = pulp.LpVariable('x1', lowBound=25)
    x2 = pulp.LpVariable('x2', lowBound=15)
    x3 = pulp.LpVariable('x3', lowBound=20)
    x4 = pulp.LpVariable('x4', lowBound=10)
    
    # Mục tiêu 1: GDP tăng tối đa
    f1 = 0.85*x1 + 1.20*x2 + 0.95*x3 + 1.35*x4
    # Mục tiêu 2: Bình đẳng / An sinh xã hội (Ưu tiên nhân lực x3 và giảm thiểu đầu tư công nghệ phân cực x2)
    # f2 = x3 - 0.5*x2
    f2 = 1.5*x3 - 0.5*x2
    
    prob += w1 * f1 + w2 * f2, "Weighted_Objective"
    prob += x1 + x2 + x3 + x4 <= budget, "Budget"
    prob += x2 + x4 >= 0.35 * (x1 + x2 + x3 + x4), "Strategic"
    
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    x_vals = np.array([x1.varValue, x2.varValue, x3.varValue, x4.varValue])
    # Tính giá trị thực tế của từng hàm mục tiêu
    val_f1 = 0.85*x_vals[0] + 1.20*x_vals[1] + 0.95*x_vals[2] + 1.35*x_vals[3]
    val_f2 = 1.5*x_vals[2] - 0.5*x_vals[1]
    
    return x_vals, val_f1, val_f2

# Quét qua dải trọng số từ 0 đến 1
w1_grid = np.linspace(0.0, 1.0, 50)
f1_results = []
f2_results = []
allocations = []

for w1 in w1_grid:
    x, f1_val, f2_val = solve_weighted_multiobjective(w1)
    f1_results.append(f1_val)
    f2_results.append(f2_val)
    allocations.append(x)

f1_results = np.array(f1_results)
f2_results = np.array(f2_results)
allocations = np.array(allocations)

## 2. Trực Quan Hóa Biên Pareto (Pareto Frontier)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(f1_results, f2_results, c=w1_grid, cmap='viridis', s=50, zorder=3)
plt.plot(f1_results, f2_results, color='grey', linestyle='--', alpha=0.5)
plt.title('Biên tối ưu Pareto (Pareto Frontier) của mô hình phân bổ ngân sách')
plt.xlabel('Hàm mục tiêu F1: Tăng trưởng GDP (Cực đại)')
plt.ylabel('Hàm mục tiêu F2: Bình đẳng & An sinh xã hội (Cực đại)')
cbar = plt.colorbar()
cbar.set_label('Trọng số w1 (Ưu tiên tăng trưởng)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Điểm Thỏa Hiệp Tối Ưu Nash (Nash Bargaining Solution)
Điểm Nash Bargaining Solution cực đại hóa tích số các lợi ích vượt trội so với điểm tham chiếu tối thiểu (disagreement point).

In [ ]:
# Xác định điểm disagreement (giá trị tối thiểu có thể chấp nhận của F1 và F2)
f1_min, f2_min = np.min(f1_results), np.min(f2_results)

# Cực đại hóa Nash Product: (F1 - F1_min) * (F2 - F2_min)
nash_product = (f1_results - f1_min) * (f2_results - f2_min)
best_idx = np.argmax(nash_product)

print(f"Điểm thỏa hiệp Nash tối ưu tại chỉ số trọng số w1 = {w1_grid[best_idx]:.3f}:")
print(f"- GDP tăng thêm F1*: {f1_results[best_idx]:.2f} nghìn tỷ VND")
print(f"- Điểm an sinh xã hội F2*: {f2_results[best_idx]:.2f}")
print(f"- Phân bổ ngân sách tối ưu: Infra={allocations[best_idx][0]:.2f}, AI={allocations[best_idx][1]:.2f}, Human={allocations[best_idx][2]:.2f}, R&D={allocations[best_idx][3]:.2f}")

## 4. Thảo Luận Chính Sách (3-5 câu mỗi câu hỏi)

**Câu a: Giải thích khái niệm tập tối ưu Pareto và ý nghĩa của biên Pareto trong việc hỗ trợ ra quyết định kinh tế.**
*Trả lời:* Tập tối ưu Pareto đại diện cho tập hợp tất cả các phương án phân bổ ngân sách mà ở đó chúng ta không thể cải thiện bất kỳ một mục tiêu nào (ví dụ tăng trưởng GDP) nếu không làm suy giảm đi ít nhất một mục tiêu khác (như công bằng xã hội). Biên Pareto vẽ ra ranh giới giới hạn năng lực kinh tế tối đa của hệ thống, giúp loại bỏ hoàn toàn các quyết định phân bổ kém hiệu quả nằm sâu phía trong. Dựa vào biên Pareto, các nhà hoạch định chính sách có được một cái nhìn trực quan toàn diện để cân nhắc sự đánh đổi một cách tường minh. Thay vì tìm kiếm một giải pháp hoàn hảo duy nhất không tồn tại, họ có thể chọn lựa phương án thỏa hiệp tối ưu nhất dựa trên định hướng chính trị của từng thời kỳ.

**Câu b: Điểm thỏa hiệp Nash (Nash Bargaining Solution) giúp giải quyết xung đột lợi ích giữa các bên liên quan như thế nào?**
*Trả lời:* Điểm thỏa hiệp Nash đóng vai trò là một trọng tài toán học khách quan giúp hài hòa hóa quyền lợi giữa các nhóm lợi ích có mục tiêu xung đột nhau trong nền kinh tế. Bằng cách cực đại hóa tích số vượt trội của lợi ích so với điểm tham chiếu tối thiểu (disagreement point), điểm Nash bảo đảm rằng không một nhóm nào bị chèn ép quá mức và mỗi bên đều nhận được một phần chia sẻ lợi ích công bằng nhất. Giải pháp này hạn chế tối đa các tranh chấp và xung đột quyền lực trong quá trình phân bổ ngân sách công. Điều này tạo điều kiện thuận lợi để đạt được sự đồng thuận cao của các bộ ngành trong thực tiễn điều hành vĩ mô.

**Câu c: Khi ưu tiên mục tiêu an sinh xã hội (F2 tăng), phân bổ cho hạng mục đào tạo nhân lực (x3) và phát triển AI (x2) thay đổi ra sao?**
*Trả lời:* Khi trọng tâm chính sách dịch chuyển mạnh mẽ sang mục tiêu bình đẳng và an sinh xã hội (F2 tăng), dòng vốn đầu tư ghi nhận sự tái cơ cấu sâu sắc. Hạng mục đào tạo nhân lực (x3) nhận được lượng vốn tăng vọt do đây là công cụ trực tiếp giúp nâng cao năng lực tự thân và thu nhập của người lao động. Ngược lại, nguồn ngân sách phân bổ cho phát triển AI (x2) bị kiềm chế hoặc giảm nhẹ để hạn chế tối đa các tác động tiêu cực của làn sóng tự động hóa gây sa thải lao động quy mô lớn. Sự điều chỉnh này tạo ra bộ đệm an toàn giúp nền kinh tế chuyển đổi số một cách êm ái hơn.